<div class="rqT", title="📥 README first">

[**Click here: Local (cloning) *vs*. Google Colab**](https://github.com/rpoteau/pyPhysChem/blob/main/colab_vs_local.md)

if you choose a Google Colab cloud session, click on this icon (unless *you are already* in Google Colab):

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rpoteau/pyPhysChem/blob/main/ULR_DigitalChem1.ipynb) 

</div>

In [ ]:
import requests

init_url = "https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/colab_init.py"
try:
    r = requests.get(init_url, timeout=5)
    r.raise_for_status()
except requests.exceptions.RequestException:
    print("⚠️ Offline or connection error. Local setup assumed.")
else:
    exec(r.text)
    setup_environment(ML=False,clone_repo=False)

<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

Entre autres choses, la cellule d'initialisation en début de notebook (celle qui exécute `colab_init.py`) rend disponible une **fonction `get_data()`**.

`get_data(...)` est conçue pour lire des fichiers de données dans n'importe quel environnement de travail, y compris Google Colab. Elle va chercher le fichier dont vous avez besoin : s'il est déjà présent localement (par exemple si vous avez cloné le dépôt `pyPhysChem`), elle l'utilise directement ; sinon, elle le télécharge automatiquement depuis GitHub (le dépôt `pyPhysChem` par défaut). Vous n'avez rien à faire de particulier : utilisez-la simplement à la place du chemin du fichier.

- **Avec `pandas`**, pour lire un fichier Excel fourni en ligne sur le GitHub de `pyPhysChem` :

    ```python
    import pandas as pd
    df = pd.read_excel(get_data("ML-data/Titration/Titration_KMnO4.xlsx"), sheet_name="nom de l'onglet")
    display(df)
    ```

<br>

- **Avec `open`**, pour lire un fichier texte (ici un fichier Molfile) :

    ```python
    with open(get_data("Molecules-data/anisole.mol")) as f:
        anisole_mol = f.read()
    ```

<br>

- **Pour importer votre propre fichier** (au lieu de récupérer celui du dépôt GitHub), utilisez l'option `source="colab"` : un bouton d'envoi de fichier s'affiche (disponible uniquement sur Google Colab). Aucun nom de fichier n'est à fournir, la fonction renvoie le nom du fichier que vous avez choisi :
    
    ```python
    df = pd.read_excel(get_data(source="colab"))
    ```

</div>

In [ ]:
import pyphyschemtools as t4pPC
from pyphyschemtools import fg, hl, bg, color
t4pPC.init()
%matplotlib inline

__author__ = "Romuald Poteau, LPCNO (INSA-CNRS-UT lab, Toulouse, France)"

import numpy as np
from matplotlib import pyplot as plt

# Veillez à ce que ces deux lignes soient commentées sur Google Colab, sinon vous aurez une erreur
%load_ext autoreload
%autoreload 2

# Université de La Réunion · Licence de Chimie · Sciences des données - Chimie physique et théorique - 1

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/MiscImages/logo_univ_reunion_DigitalChem.png" width="1000">
</div>

<br>

<u>**Au programme**</u>

1. **Constantes fondamentales et unités** : `scipy.constants`, unité de masse atomique, masses molaires, préfixes
2. **Atomistique** : taille des atomes, isotopes et datation au carbone 14, atome d'hydrogène et ions hydrogénoïdes (niveaux d'énergie, séries spectrales), orbitales atomiques
3. **Tableau périodique** : structure, évolution des propriétés
4. **Traitement statistique de données** : tableaux de données avec `pandas`, régression, droites d'étalonnage, données aberrantes
5. **Représentations moléculaires** : le format SMILES, structures 3D
6. **Du SMILES à la structure de Lewis** : doublets non liants, aromaticité, hétérocycles
7. **Symétrie moléculaire** : opérations de symétrie, groupes ponctuels, symétrie des orbitales
8. **Chimie orbitalaire des systèmes π** : méthode de Hückel, conjugaison, aromaticité (règle de Hückel), symétrie des OM π

<u>**Les outils**</u>

* **numpy** : calcul numérique sur des tableaux de nombres
* **matplotlib** et **seaborn** : tracé de graphiques
* **scipy** : constantes physiques, ajustements, outils scientifiques
* **pandas** : manipulation de tableaux de données
* **RDKit** : une bibliothèque de chimie informatique, qui « connaît » la chimie (SMILES, structures de Lewis, aromaticité...)
* **mendeleev** : base de données des éléments et de leurs isotopes
* **pyphyschemtools** : une bibliothèque développée pour cet enseignement, qui simplifie l'utilisation de ces outils (visualisation de molécules, atome d'hydrogène, symétrie, méthode de Hückel, mise en forme, etc.)


<u>**Mode d'emploi**</u>

- Pour une thématique donnée, exécutez les cellules **dans l'ordre**.
- Les exercices sont signalés par un encadré. Complétez le code et **répondez aux questions par une phrase** : un résultat numérique n'a de sens qu'avec son unité et son interprétation chimique.
- En cas d'erreur, lisez le message : il indique presque toujours la ligne fautive.
</div>

## [A] Constantes fondamentales, unités

### [A.1] À la découverte de `scipy.constants`

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

On va se servir du module `scipy.constants`.

Il donne accès aux valeurs recommandées par le *Committee on Data of the International Science Council* (<a href="https://en.wikipedia.org/wiki/Committee_on_Data_of_the_International_Science_Council">CODATA</a>) pour les constantes physiques fondamentales (constante de Planck, charge élémentaire, masse de l'électron, nombre d'Avogadro, etc.). Les utiliser plutôt que des valeurs codées en dur garantit des calculs cohérents, précis et à jour, et évite les erreurs de recopie.

Chaque constante est accessible par son nom via `sc.value("...")` pour la valeur seule, ou via `sc.physical_constants["..."]` qui renvoie le triplet **(valeur, unité, incertitude)**.


</div>
<br>
<div class='ex'>

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.1]</b></span></td></tr></table>

1. Analysez le code python ci-dessous, et vérifiez que vous comprenez ce que fait chaque ligne du code.
2. Commentez ces résultats : nombre de chiffres significatifs, unité, incertitude. Pourquoi l'incertitude sur le nombre d'Avogadro est-elle nulle ? (vous avez le droit d'allez consulter wikipedia).
</div>

In [ ]:
# Démo

import scipy.constants as sc
print(f"vitesse de la lumière: {sc.physical_constants['speed of light in vacuum']}")
print()
print(f"charge élémentaire: {sc.physical_constants['elementary charge']}")
print(f"masse de l\'électron: {sc.physical_constants['electron mass']}")
print(f"masse du neutron : {sc.physical_constants['neutron mass']}")
print()
print(f"constante des gaz parfaits: {sc.physical_constants['molar gas constant']}")

R = sc.value('molar gas constant')
print(R)
me = sc.value('electron mass')
print(me)

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.2]</b></span></td></tr></table>

1. Affichez le triplet *(valeur, unité, incertitude)*
    - du nombre d'Avogadro (`"Avogadro constant"`)
    - de la masse du proton (`"proton mass"`)
2. Stockez la valeur de ces deux grandeurs dans des variables `NA` et `mp`. Calculez ensuite le rapport masse du proton / masse de l'électron. Que représente ce nombre ? Formulez une phrase du type "*un proton est environ … fois plus ... qu'un électron*".
3. Combien y a-t-il de molécules d'eau dans une goutte d'eau pure de 5 μL, sachant que *M*<sub>H2O</sub> = 18.015 g.mol<sup>-1</sup> et *&rho;*<sub>H2O</sub> = 1 g.mL<sup>-1</sup>. Afficher le résultat en notation scientifique, avec 1 décimale.
</div>

In [ ]:
# Entrez le code python



### [A.2] unités de masse atomique unifiée (u)

#### [A.2.1] Définition, relation u ↔ g.mol<sup>-1<sup>

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

La masse d'un atome s'exprime en **unité de masse atomique unifiée** (u), définie comme $\frac{1}{12}$ de la masse d'un atome de carbone 12.

Numériquement, la masse molaire d'un élément en g·mol<sup>-1</sup> est égale à sa masse atomique en u : ainsi le carbone 12, de masse 12 u, a une masse molaire de ≈12 g·mol<sup>-1</sup>, à mieux que $10^{-9}$ près.

Cette correspondance u ↔ g·mol<sup>-1</sup> (à $10^{-9}$ près) découle de la définition de la mole ($M = N_A \times m_{\text{atome}}$) ; l'écart à l'égalité exacte entre u et masse molaire provient de la révision du SI de 2019 qui fixe désormais $N_A$ et la masse de l'uma indépendamment.

Pour les applications numériques, on va à nouveau se servir du module `scipy.constants`. 

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.3]</b></span></td></tr></table>

Analysez et appropriez-vous le code ci-dessous

</div>

In [ ]:
# Démo

import scipy.constants as sc
print(f'uma: {sc.physical_constants["atomic mass constant"]}')
print(f'constante de masse molaire: {sc.physical_constants["molar mass constant"]}')
print(f'NA: {sc.physical_constants["Avogadro constant"]}')

print()
uma = sc.value("atomic mass constant")
NA = sc.value("Avogadro constant")
print("uma * NA = ", uma*NA,"kg.mol-1 = ",uma*NA*1000,"g.mol-1")

#### [A.2.2] Masses molaires

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.4]</b></span></td></tr></table>

1. La caféine a une masse moléculaire de 194,19 u. Calculez la masse d'**une** molécule de caféine en kg, puis sa masse molaire en g/mol. Attention aux chiffres significatifs ! Que constatez-vous ?
2. Le résultat de `uma * NA` vaut-il exactement 1 g/mol ? Combien de chiffres significatifs faut-il pour voir la différence ? Cette différence a-t-elle une importance pour un chimiste ? Qu'en concluez-vous d'un point de vue pratique

</div>

In [ ]:
# Entrez le code python



### [A.3] Préfixes

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Les préfixes définis dans `scipy.constants`, de 10<sup>-30</sup> à 10<sup>+30</sup>. 
</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.5]</b></span></td></tr></table>

Analysez le code ci-dessous. Vous allez voir qu'on se sert aussi des  **pyPhysChemTools**, qui vont nous accompagner tout au long de ces exercices.
</div>

In [ ]:
# Démo

from pyphyschemtools import PhysicalQuantity

PhysicalQuantity.show_available_prefixes()

import scipy.constants as sc
print()
print("1 quintillionième = ",sc.quecto)

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [A.6]</b></span></td></tr></table>

Voici deux énergies caractéristiques, en J :
- Énergie de un photon visible = 3.0e-19 J (λ ≈ 660 nm)
- Énergie d'une liaison C-C = 5.8e-19  J

1. Ré-exprimez chacune de ces deux énergies en **attojoules** puis en **zeptojoules**, à l'aide des valeurs numériques des préfixes définies dans `scipy.constants`. Affichez la valeur suivie de l'unité exprimée avec son diminutif.
2. Pour chaque énergie, indiquez quel préfixe donne la valeur la plus lisible (mantisse entre 1 et 1000).
3. Calculez le rapport `E_CC / E_photonV`. Combien de fois la liaison C–C est-elle plus énergétique qu'un photon visible ?
4. L'énergie d'un photon UV (λ ≈ 300 nm) vaut environ `E_photonUV = 6.6e-19` J. Calculez le rapport `E_photonUV / E_liaison`. Un photon UV est-il capable de rompre une liaison C–C ? Comparez avec le cas du photon visible.

</div>

In [ ]:
# Entrez le code python



### [A.4] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [B] Atomistique

### [B.1] Volume et rayon d'un atome

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.1]</b></span></td></tr></table>

Les atomes, à l’état solide, ont un volume molaire de l’ordre de 5 cm<sup>3</sup> .mol<sup>−1</sup>
1. Quels sont, approximativement, les volume et rayon d’un atome ?
2. Sachant que le diamètre d’un noyau est environ 100 000 fois inférieur à celui d’un atome, calculer le rapport entre le volume d’un atome et le volume de son noyau.
</div>

In [ ]:
# Entrez le code python



### [B.2] Isotopes

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Rappel de cours : l'isotopie**</u></span>

**Le noyau atomique.** Le noyau d'un atome contient des protons et des neutrons, appelés ensemble **nucléons**. On le caractérise par deux nombres :
- le **numéro atomique** Z, qui est le nombre de protons. Il définit l'élément chimique : tous les atomes de carbone ont Z = 6 ;
- le **nombre de masse** A, qui est le nombre de nucléons. Le nombre de neutrons vaut donc N = A − Z.

Un noyau donné, appelé **nucléide**, se note $^{A}_{Z}\mathrm{X}$, par exemple $^{12}_{6}\mathrm{C}$. Comme Z est fixé par le symbole de l'élément, on écrit souvent simplement $^{12}\mathrm{C}$ (carbone 12).

**Définition.** Des **isotopes** sont des atomes d'un même élément (même Z) qui diffèrent par leur nombre de neutrons, donc par leur nombre de masse A. Par exemple :
- hydrogène : $^{1}\mathrm{H}$ (0 neutron), deutérium $^{2}\mathrm{H}$ = D (1 neutron), tritium $^{3}\mathrm{H}$ = T (2 neutrons) ;
- carbone : $^{12}\mathrm{C}$, $^{13}\mathrm{C}$, $^{14}\mathrm{C}$ ;
- chlore : $^{35}\mathrm{Cl}$, $^{37}\mathrm{Cl}$.

**Propriétés.** Les isotopes d'un élément ont le même nombre d'électrons et la même configuration électronique : leurs **propriétés chimiques** sont pratiquement identiques. Ils diffèrent par leur **masse**, ce qui se voit :
- en spectrométrie de masse, par exemple dans les massifs isotopiques du chlore et du brome ;
- en spectroscopie de vibration : une liaison C–D vibre à plus basse fréquence qu'une liaison C–H ;
- en RMN, où seuls certains isotopes sont observables, comme $^{1}\mathrm{H}$ et $^{13}\mathrm{C}$, mais pas $^{12}\mathrm{C}$.

Certains isotopes sont stables, d'autres sont **radioactifs** : c'est le cas de $^{14}\mathrm{C}$, utilisé pour la datation, et du tritium.

**Abondance naturelle et masse molaire.** Dans la nature, un élément est un mélange de ses isotopes stables, en proportions à peu près constantes, appelées **abondances isotopiques**. La masse molaire atomique indiquée dans le tableau périodique est la moyenne des masses molaires $M_i$ des isotopes, pondérée par leurs abondances $x_i$ :

$$M = \sum_i x_i \, M_i$$

Par exemple, pour le chlore (75,76 % de $^{35}\mathrm{Cl}$ et 24,24 % de $^{37}\mathrm{Cl}$) :

$$M(\mathrm{Cl}) = 0{,}7576 \times 34{,}969 + 0{,}2424 \times 36{,}966 \approx 35{,}45\ \mathrm{g\,mol^{-1}}$$

Quelques éléments n'ont qu'un seul isotope stable : on les appelle **éléments mononucléidiques**. C'est le cas de F, Na, Al, P, Co ou Au.

</div>

#### [B.2.1] Isotopes de Li

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.2]</b></span></td></tr></table>

Le lithium naturel est un mélange des deux isotopes <sup>6</sup>Li et <sup>7</sup>Li dont les masses molaires sont respectivement 6.015 et 7.016 g.mol<sup>−1</sup>. Sa masse molaire apparente (lithium à l’état naturel) est de 6,94 g.mol<sup>−1</sup>. On veut savoir quel est en pourcentage sa composition isotopique. Pour cela, on a écrit le code ci-dessous. L'analyser pour identifier ce qui a été fait, et écrire l(es) équation(s) correpondante(s).

*Donnée*: [documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) de `numpy.linalg.solve`.
</div>

In [ ]:
# Démo

t4pPC.centerTitle("Lithium")
M6 = 6.015
M7 = 7.016
M = 6.94

A = np.array([
             [M6,M7],
             [1,1]
             ])
y = np.array([M,1])
print(A)
print()
print(y)
p = np.linalg.solve(A, y)
print()
print(p[0]*100)
print(p[1]*100)

#### [B.2.2] Isotopes de Cl

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.3]</b></span></td></tr></table>

Le chlore naturel, de masse atomique relative 35.453 u, est un mélange de deux isotopes <sup>35</sup>Cl et <sup>37</sup>Cl.

(a) Exploiter le code donné pour Li pour calculer les proportions respectives des isotopes isotopes <sup>35</sup>Cl et <sup>37</sup>Cl.

(b) Quelle est la masse moyenne du dichlore (Cl<sub>2</sub>) ?

(c) Il n’existe pas d’atome avec une masse atomique égale à 35.459 u de même qu’il n’existe pas non
plus de molécule Cl<sub>2</sub> de masse 70.906. Le dichlore est un mélange de molécules ayant les diverses
compositions isotopiques possibles. Combien existent-il de types différents de molécules Cl<sub>2</sub> et
quelle est leur masse ?

(d) Quelles sont les proportions relatives des différentes molécules de Cl<sub>2</sub> ?

(e) Calculer la masse du dichlore à partir de la moyenne pondérée des masses moléculaires des
différents types de molécules de Cl<sub>2</sub> . Quelle remarque pouvez-vous faire ?

*Données* : Masse atomique du <sup>35</sup>Cl : 34.969 u ; Masse atomique du <sup>37</sup>Cl : 36.966 u

</div>

In [ ]:
# Entrez le code python



#### [B.2.3] Listes des isotopes d'un élément

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

On va à nouveau se servir de la bibliothèque `pyPhysChemTools`, et plus précisément de sa classe `TableauPeriodique`, qui s'appuie sur la bibliothèque `mendeleev` pour aller chercher les données chimiques (isotopes, abondances, demi-vies, masses...). Voici un exemple d'utilisation ci-dessous.

```python
from pyphyschemtools import TableauPeriodique

tp = TableauPeriodique()
tp.isotopes("Li") 
```

<br>

Toutes les fonctionnalités sont décrites dans la [documentation du module `PeriodicTable`, qui contient la **classe `TableauPeriodique`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/PeriodicTable.html).

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.4]</b></span></td></tr></table>

1. Utiliser `pyPhysChemTools` pour retrouver les isotopes stables du chlore.
2. Comparer les valeurs d'abondance que vous avez calculées avec celles de la bibliothèque `mendeleev`.
</div>

In [ ]:
# Entrez le code python



#### [B.2.4] Datation au carbone 14

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Le carbone possède un isotope radioactif, le carbone 14, utilisé pour dater des matières organiques (méthode de datation au carbone 14, due à [Willard Libby](https://fr.wikipedia.org/wiki/Willard_Frank_Libby), qui reçut le Prix Nobel de Chimie en 1960 "pour sa méthode d'utilisation du carbone 14 servant à déterminer l'âge en archéologie, en géologie, en géophysique et d'autres branches de la science"). 

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.5]</b></span></td></tr></table>

1. Utiliser le module `TableauPeriodique` de `pyphyschemtools` pour lister les isotopes du carbone. Repérer dans le tableau l'isotope radioactif du carbone utilisé pour la datation, et noter sa demi-vie $t_{1/2}$ (attention à l'unité).
2. En déduire la constante de décroissance radioactive $\lambda$.
3. Exprimer $t$ en fonction de $N_0$, $N(t)$ et $\lambda$.
4. Un fragment d'os contient aujourd'hui 68 % de la quantité de <sup>14</sup>C qu'il contenait à la mort de l'organisme. Calculer son âge.
5. Écrire une fonction Python `age_carbone14(fraction_restante)` qui renvoie cet âge (en années) pour n'importe quelle fraction restante, et vérifier votre calcul de la question 4 avec cette fonction.
6. *Pour aller plus loin* : tracer $N(t)/N_0$ en fonction du temps (0 à 50 000 ans), et superposer la valeur trouvée à la question 4.

*Données:* La décroissance radioactive suit une loi cinétique d'ordre 1 (vue en cinétique) : $\dfrac{dN}{dt} = -\lambda N$, dont la solution est $N(t) = N_0\,e^{-\lambda t}$.

Par définition, la demi-vie $t_{1/2}$ est le temps au bout duquel $N(t_{1/2}) = N_0/2$ :

$$\frac{N_0}{2} = N_0\,e^{-\lambda t_{1/2}} \quad\Longrightarrow\quad e^{-\lambda t_{1/2}} = \frac{1}{2} \quad\Longrightarrow\quad \lambda = \frac{\ln 2}{t_{1/2}}$$

C'est la même relation que celle vue pour les cinétiques d'ordre 1 en chimie (temps de demi-réaction).

- ⚠️ Attention au vocabulaire anglais utilisé par `numpy` : `np.log()` calcule le logarithme **népérien** $\ln$ (et non le logarithme décimal !). Le logarithme décimal $\log_{10}$ correspond à `np.log10()`.
- ⚠️ ne pas utiliser `lambda` comme variable ou fonction, c'est une fonction spéciale en python qui ne doit pas être modifiée.
</div>

In [ ]:
# Entrez le code python



### [B.3] Atome d'hydrogène et ions hydrogénoïdes

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
    
Le module `hydrogen.py` de `pyphyschemtools` implémente le modèle de Bohr /
la solution analytique de l'équation de Schrödinger pour l'atome d'hydrogène
et les ions hydrogénoïdes (systèmes à un seul électron : H, He<sup>+</sup>, Li<sup>2+</sup>...).

Dans ce contexte, les équations majeures sont :

- Relation de Planck-Einstein : $E = h\nu = \dfrac{hc}{\lambda}$

- Niveaux d'énergie (solution de l'équation de Schrödinger) : $E_n = -hcR\dfrac{Z^2}{n^2}$

- Formule de Rydberg (transition $n_2 \to n_1$) : $\dfrac{1}{\lambda} = R\,Z^2\left(\dfrac{1}{n_1^2} - \dfrac{1}{n_2^2}\right)$

Le module `hydrogen.py` fournit :
- `rydberg_Z` : la constante de Rydberg, corrigée de la masse finie du noyau
  (masse réduite) ;
- `En_hydrogenic` : l'énergie d'un niveau `n` (en joules) ;
- `Ei_hydrogenic` : l'énergie d'ionisation (en eV) ;
- `spectral_series` : l'affichage des séries spectrales usuelles (Lyman,
  Balmer, Paschen...) avec leurs énergies, longueurs d'onde et limites.

Toutes les fonctionnalités sont décrites dans la [documentation du module `Hydrogen`, qui contient **plusieurs fonctions**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Hydrogen.html).

> **Remarque :** ce module couvre les niveaux d'énergie, les énergies
> d'ionisation et les spectres (raies et diagrammes). Les orbitales
> atomiques elles-mêmes — parties radiale et angulaire, et leur
> visualisation 2D/3D — sont dans le module séparé `AO.py`.

</div>

#### [B.3.1] Prise en main

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.6]</b></span></td></tr></table>

1. Importer le module `hydrogen.py`.
2. Afficher la docstring de chacune de ses fonctions publiques (`rydberg_Z`, `En_hydrogenic`, `Ei_hydrogenic`, `spectral_series`), par exemple avec `help()`, ou bien consultez la version en ligne, qu'on appelle la référence de l'API `hydrogen.py`.
</div>

In [ ]:
# Entrez le code python



#### [B.3.2] Atome H

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.7]</b></span></td></tr></table>

1. Calculer l'énergie de son état fondamental, en eV puis en joules.
2. Calculer l'énergie d'ionisation de l'atome d'hydrogène, en eV, avec 5 chiffres significatifs.
3. Afficher, en notation scientifique, la valeur de la constante de Rydberg avec et sans la correction de masse réduite.
4. Comparer les énergies d'ionisation avec et sans la correction de masse réduite
  (`reduced_mass=True`/`False`). Commenter l'ordre de grandeur de l'écart. Quelle est la valeur la plus précise ?
</div>

In [ ]:
# Entrez le code python



#### [B.3.3] Énergie d'ionisation de H et des ions hydrogenoïdes

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.8]</b></span></td></tr></table>

1. Calculer l'énergie d'ionisation des ions hydrogénoïdes He<sup>+</sup>, Li<sup>2+</sup>, Be<sup>3+</sup>...
  jusqu'à Z=10 (Ne<sup>9+</sup>).
2. Tracer l'énergie de l'état fondamental E1s (en eV) en fonction de *Z*. Commenter : que se passe-t-il quand *Z* augmente ? Pourquoi devient-il de plus en plus difficile d'arracher l'électron ?
</div>

In [ ]:
# Entrez le code python



#### [B.3.4] Énergies de transition de H

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.9]</b></span></td></tr></table>

1. Calculer toutes les énergies de transition (en eV) et les longueurs d'onde de transition (en nm) des [séries de Lyman, Balmer, Paschen, Brackett, Pfund et Humphreys](https://fr.wikipedia.org/wiki/Spectre_de_l%27atome_d%27hydrog%C3%A8ne) de l'hydrogène, pour n variant de 1 à 12.
2. Que représente `lambda_inf`, pour chaque série ?
</div>

In [ ]:
# Entrez le code python



#### [B.3.5] Diagrammes d'énergie et spectres simulés

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.10]</b></span></td></tr></table>

1. Affichez l'aide de la fonction `hydrogen.plot_energy_levels` et repérez :
    - ce que renvoie la fonction.
    - le paramètre qui choisit entre absorption et émission ;
2. Un atome d'hydrogène dans son état fondamental reçoit une énergie de 13.221 eV. Dans quel état se retrouve-t-il ?
3. Tracez tous les chemins de désexcitation possibles d'un atome d'hydrogène excité dans cet état. Que représentent Ly, Ba, etc. ?
4. Tracez le spectre d'absorption de l'état fondamental vers *n*<sub>2</sub> = 15, en ne reportant sur l'axe des ordonnées que les énergies des 5 premiers niveaux.

</div>

In [ ]:
# Entrez le code python



### [B.4] Orbitales Atomiques

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Résoudre l'équation de Schrödinger pour l'atome d'hydrogène donne une famille de fonctions d'onde ψ, les **orbitales atomiques** (OA). Chacune est repérée par trois nombres quantiques :

- *n* (principal) : fixe l'énergie et la taille de l'OA ;
- *ℓ* (secondaire) : fixe la forme de l'OA (s, p, d, f pour *ℓ* = 0, 1, 2, 3) ;
- *m*<sub>ℓ</sub> (magnétique) : fixe son orientation dans l'espace.

Ces nombres ne sont pas indépendants : 0 ≤ *ℓ* ≤ *n* − 1 et −*ℓ* ≤ *m*<sub>ℓ</sub> ≤ +*ℓ*.

Une OA est une fonction ψ(*r*, *θ*, *φ*) qui prend une valeur en chaque point de l'espace, repéré par ses coordonnées sphériques. Le module `AO` de `pyphyschemtools` propose deux représentations complémentaires (`plot_orbital_3D` & `plot_electron_cloud`), que l'on va comparer.

Votre code Python doit débuter par :

```python
from pyphyschemtools import AO
```
</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [B.11]</b></span></td></tr></table>

On considère les triplets (*n*, *ℓ*, *m*<sub>ℓ</sub>) suivants : (1,0,0), (2,1,0), (3,2,0), (3,2,1), (3,3,0).

1. Pour chaque triplet, indiquez à quelle sous-couche (1s, 2p, 3d…) appartient l'orbitale atomique (OA) de l'hydrogène correspondante. L'un de ces triplets ne correspond à aucune OA : lequel, et pourquoi ?
2. Affichez l'aide des fonctions `plot_orbital_3D` & `plot_electron_cloud` (commande : `help(nom de la fonction)`
3. Représentez les OA valides avec la fonction `plot_orbital_3D` du module `AO` de `pyphyschemtools`.
4. Que représentent les surfaces grises ? Quel sens faut-il donner à ces surfaces ?
5. Pour les OA telles que *ℓ* ≥ 1, déduisez de la représentation de quelle OA il s'agit : 2p<sub>x</sub>, 2p<sub>y</sub>, 2p<sub>z</sub>, 3d<sub>xy</sub>, 3d<sub>xz</sub>, 3d<sub>yz</sub>, 3d<sub>x<sup>2</sup>-y<sup>2</sup></sub>, 3d<sub>z<sup>2</sup></sub>…
6. Utilisez la fonction `plot_electron_cloud` du module `AO` pour tracer l'OA 1s, puis l'OA 3d<sub>yz</sub>. Qu'est-ce qui est représenté ici ? En quoi cette représentation diffère-t-elle de celle obtenue avec `plot_orbital_3D` ?

</div>

In [ ]:
t4pPC.centerTitle("2. help")
# Entrez le code python



In [ ]:
t4pPC.centerTitle("3. Représentation 3D des OA de l'atome d'hydrogène")
# Entrez le code python



In [ ]:
t4pPC.centerTitle("6. Nuage électronique dans un plan")
# Entrez le code python



### [B.5] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [C] Tableau périodique

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Proposé par Dmitri Mendeleïev en 1869, réorganisé par numéro atomique par Henry Moseley en 1913, puis formulé dans sa forme actuelle par Glenn Seaborg en 1944, le tableau périodique classe les éléments chimiques par **numéro atomique *Z*** croissant. Sa spécificité, qui en fait un atout sur le plan physico-chimique, est de regrouper les éléments aux propriétés chimiques semblables :

- une **ligne** (période) correspond au remplissage d'une couche électronique, de numéro *n* ;
- une **colonne** (groupe) rassemble des éléments ayant la même configuration électronique de valence, et donc des propriétés chimiques voisines ;
- les **blocs** s, p, d et f indiquent la sous-couche en cours de remplissage.

Les propriétés des éléments (rayon, énergie d'ionisation, électronégativité…) varient ainsi de façon **périodique** avec *Z* : c'est ce qu'on va observer.

La bibliothèque `mendeleev` rassemble les propriétés de tous les éléments chimiques ainsi que la structure du tableau périodique (position de chaque élément, période, groupe, bloc). `pyphyschemtools` en propose une version simplifiée, sous forme d'un tableau périodique interactif.

Son utilisation se fait en deux étapes :

```python
from pyphyschemtools import TableauPeriodique

tp = TableauPeriodique()   # 1. on crée le tableau et on le stocke dans la variable tp
tp.afficher()              # 2. on l'affiche
```

- `TableauPeriodique()` crée un objet tableau périodique (on parle d'**instanciation**), qui contient toutes les données des éléments. On le range dans une variable, ici `tp`, pour pouvoir s'en resservir ensuite.
- `tp.afficher()` demande à cet objet de **s'afficher**. Survolez un élément avec la souris pour voir ses propriétés.

Les données elles-mêmes sont accessibles sous forme de tableau avec `tp.elements` : c'est ce qu'on utilisera pour tracer des propriétés en fonction de *Z*.

Toutes les fonctionnalités sont décrites dans la [documentation du module `PeriodicTable`, qui contient la **classe `TableauPeriodique`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/PeriodicTable.html).

</div>


### [C.1] Structure du tableau périodique

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [C.1]</b></span></td></tr></table>

On va commencer par revoir le tableau périodique.

1. Afficher le tableau périodique interactif de pyphyschemtools.
1. Pourquoi les éléments de la colonne du zinc (Zn, Cd, Hg) n'ont-ils pas la même couleur que les métaux de transition, affichés quant à eux sur un fond de couleur vieux rose clair ?
2. Pourquoi l'hydrogène n'a-t-il pas la même couleur que les autres éléments de la première colonne ?
3. Comparez la configuration électronique fondamentale de <sub>23</sub>V et de <sub>24</sub>Cr, puis celle de <sub>28</sub>Ni et de <sub>29</sub>Cu. Ces configurations respectent-elles la règle de Klechkowski ?
4. Repérez les métalloïdes. Comment sont-ils disposés dans le tableau ?
5. Pourquoi les lanthanides et les actinides sont-ils placés sous le tableau ?
6. Pourquoi les propriétés de <sub>117</sub>Ts sont-elles inconnues (NaN = *Not a Number*) ?
7. L'hélium possède deux électrons de valence, comme les éléments du groupe 2. Pourquoi n'appartient-il pas à ce groupe ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

**Donnée.** Les nombres d'oxydation courants d'un élément s'obtiennent avec la bibliothèque `mendeleev` :

```python
from mendeleev import element
print(element("Zn").oxistates)
print(element("Fe").oxistates)
```

</div>

In [ ]:
t4pPC.centerTitle("1. Affichage du tableau périodique interactif")
# Entrez le code python



In [ ]:
t4pPC.centerTitle("Bac à sable : entrez le code nécessaire pour répondre à certain(e)s des questions")
# Entrez le code python



### [C.2] Évolution des propriétés du tableau périodique

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [C.2]</b></span></td></tr></table>

1. Exécutez le code ci-dessous et vérifiez que vous comprenez le rôle de chaque ligne. À quoi servent la variable `cm2i` et la liste `xticks` ?
2. À quels éléments correspondent les lignes verticales rouges ? Comment évolue *E*<sub>i1</sub> juste après chacune d'elles ?
3. Comment évolue *E*<sub>i1</sub> le long d'une période ? Et le long d'une colonne (comparez par exemple Li, Na, K, Rb, Cs) ? Justifier.
</div>

In [ ]:
# Démo

import matplotlib.pyplot as plt 

Z = tp.elements["atomic_number"]
Ei1 = tp.elements["Ei1"]
color = tp.elements["color"]

cm2i = 1/2.54 # conversion de cm en pouces
fig, ax = plt.subplots(figsize=(30*cm2i,12*cm2i))
xticks=[2,10,18,36,54,86,118]
ax.set_xticks(xticks)
plt.grid(color='r', linestyle='--', linewidth=0.5, axis="x")
plt.plot(Z,Ei1,linewidth=0.5)
plt.scatter(Z,Ei1,marker="o",c=color)
plt.xlabel("Numéro atomique (Z)")
plt.ylabel("Ei1 / eV")

t4pPC.save_fig("fig_examples/PeriodicTable/ENP_Z.png")
plt.show()

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [C.3]</b></span></td></tr></table>

1. Affichez la liste des propriétés disponibles avec `tp.elements.columns`. Repérez les colonnes correspondant :
    - au [rayon atomique de Rahm](https://doi.org/10.1002/chem.201700610) (= taille de l'atome isolé, calculée à partir de sa densité électronique),
    - à l'électronégativité de Pauling,
    - à l'affinité électronique.
2. En vous inspirant du code précédent, tracez sur trois graphes séparés le rayon atomique de Rahm, l'électronégativité de Pauling puis l'affinité électronique en fonction de *Z*. Pensez à adapter le titre de l'axe des ordonnées.
3. Pour certains éléments et certaines propriétés, aucun point n'est tracé. Lesquels ? Pourquoi ?
4. Justifier l'évolution globale du rayon atomique dans un groupe et dans une période. Pourquoi observe-t-on une augmentation du rayon lors du passsage du bloc d au bloc p ?
5. L'électronégativité suit la même tendance qu'une autre propriété. Laquelle ?
6. Pourquoi certaines valeurs d'affinité électronique sont-elles très négatives ?

</div>

In [ ]:
# Entrez le code python



<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [C.4]</b></span></td></tr></table>

Exécutez la celulle python ci-dessous. On a tracé la polarisabilité dipolaire $\alpha$ en fonction du rayon atomique *r*, en échelle log-log. On a ensuite tracé la température d'ébullition des gaz nobles en fonction de $\alpha$. 

*Sources des données :* les rayons atomiques et les polarisabilités proviennent de la bibliothèque `mendeleev`, qui compile des données de la littérature (ici, respectivement Rahm *et al.*, *Chem. Eur. J.* 2016, et Schwerdtfeger et Nagle, *Mol. Phys.* 2019). Les températures d'ébullition, absentes de `mendeleev`, sont tirées du *CRC Handbook of Chemistry and Physics* et saisies à la main dans le code.

1. Analysez le premier graphique.
    - Quelle règle simple en tirez-vous ?
    - Pourquoi les alcalins sont-ils nettement plus polarisables que ne le prévoit leur rayon ?
2. Analysez ensuite le tracé la température d'ébullition des gaz nobles en fonction de $\alpha$. Cette corrélation a-t-elle un sens ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

**Donnée.** Les gaz nobles ne forment pas de liaisons chimiques. À l'état liquide, leurs atomes ne sont retenus que par les **forces de dispersion de London** : le nuage électronique d'un atome fluctue en permanence, créant un dipôle instantané qui polarise ses voisins. Ces interactions sont faibles, et d'autant plus fortes que les atomes sont polarisables : l'énergie d'attraction entre deux atomes identiques est proportionnelle à $\alpha^2$.

</div>



In [ ]:
# Démo

import numpy as np
import matplotlib.pyplot as plt

d = tp.elements[["symbol", "atomic_radius_rahm", "dipole_polarizability", "block"]].dropna()
r = d["atomic_radius_rahm"].values          # pm
a = d["dipole_polarizability"].values       # a0^3

pente, b = np.polyfit(np.log10(r), np.log10(a), 1)

col = {"s": "C0", "p": "C2", "d": "C3", "f": "C4"}
fig, ax = plt.subplots(figsize=(12, 6))
for blk in "spdf":
    m = d["block"] == blk
    ax.scatter(r[m], a[m], color=col[blk], s=30, label=f"bloc {blk}")

for s, x, y in zip(d["symbol"], r, a):
    ax.annotate(s, (x, y), textcoords="offset points", xytext=(3, 3), fontsize=7)
    
xx = np.linspace(r.min(), r.max(), 100)
ax.plot(xx, 10**b * xx**pente, "k--", lw=1, label="Interpolation")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Rayon atomique de Rahm / pm")
ax.set_ylabel(r"Polarisabilité dipolaire $\alpha$ / $a_0^3$")
ax.legend()
plt.show()

###################################################################################
Teb = {"He": 4.22, "Ne": 27.07, "Ar": 87.30, "Kr": 119.93, "Xe": 165.05, "Rn": 211.5}   # K
alpha_GN = tp.elements.set_index("symbol").loc[list(Teb), "dipole_polarizability"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(alpha_GN.values, list(Teb.values()), color="C1", s=40)
for s in Teb:
    ax.annotate(s, (alpha_GN[s], Teb[s]), textcoords="offset points", xytext=(5, -3))
ax.set_xlabel(r"Polarisabilité dipolaire $\alpha$ / $a_0^3$")
ax.set_ylabel(r"$T_\mathrm{éb}$ / K")
plt.show()

### [C.3] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [D] Traitement statistique de données (sciences des données pour débutants)

### [D.1] Introduction à la bibliothèque `pandas`. Régression (alias "*fit*")

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

- **1ère partie. Lecture et analyse d'une base de données**. Le cours/TD est dans un [notebook séparé](./DS4B-Iris1.ipynb)
- **2ème partie. Régression**. Le cours/TD est dans un [notebook séparé](./DS4B-Iris2.ipynb)
</div>


### [D.2] Données chimiques, données aberrantes ("outliers"), écart au modèle

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Une mesure expérimentale n'est jamais parfaite. Lorsqu'on ajuste un modèle à des données (par exemple une droite d'étalonnage), certains points peuvent s'en écarter. Il faut alors distinguer deux situations :

- une **valeur aberrante** (outlier) : un point isolé, qui s'écarte des autres sans raison liée au modèle. C'est souvent une erreur expérimentale (dilution, cuve sale, faute de saisie). Les autres points restent bien décrits par le modèle ;
- un **écart au modèle** : plusieurs points s'écartent du modèle de façon **systématique**. Ce n'est plus une erreur de mesure, c'est le modèle lui-même qui cesse d'être valable dans cette zone.

L'outil principal pour faire la différence est le **graphe des résidus**, c'est-à-dire des écarts entre les valeurs mesurées et les valeurs calculées par le modèle : des résidus répartis au hasard autour de zéro valident le modèle, un point isolé très éloigné signale un outlier, et un motif régulier (arc, tendance) révèle un écart au modèle.
</div>

#### [D.2.1] Dosage spectrophotométrique du permanganate

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>
Le permanganate de potassium (KMnO<sub>4</sub>) donne en solution aqueuse une couleur violette intense, due à l'ion MnO<sub>4</sub><sup>−</sup>, qui absorbe fortement vers 524 nm. On peut donc le doser par spectrophotométrie, grâce à la loi de Beer-Lambert :

$$A = \varepsilon \, \ell \, c$$

où *A* est l'absorbance (sans unité), *ε* le coefficient d'absorption molaire (L.mol<sup>−1</sup>.cm<sup>−1</sup>), *ℓ* la longueur de la cuve (cm) et *c* la concentration (mol.L<sup>−1</sup>). Doser par spectrophotométrie consiste à
- mesurer l'absorbance de solutions étalons de concentrations connues,
- tracer la droite d'étalonnage *A* = f(*c*),
- s'en servir pour déterminer la concentration d'une solution inconnue.

Pour l'étalonnage, on va utiliser la classe `QuantitativeAnalysis` de `pyphyschemtools`, qui réalise l'ajustement linéaire, trace la droite et ses résidus, et calcule une concentration à partir d'une absorbance. [Sa documentation est en ligne](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Spectro.html#quantitativeanalysis-class-documentation).

Son utilisation tient en quatre lignes :

```python
from pyphyschemtools import QuantitativeAnalysis

etalonnage = QuantitativeAnalysis(c, A,                                     # c, A : listes ou colonnes de DataFrame
                                  x_label="concentration / mol.L$^{-1}$",   # titre de l'axe des abscisses
                                  y_label="absorbance")                     # titre de l'axe des ordonnées
etalonnage.fit_linear()                       # ajustement A = a.c + b, et rapport (pente, R², LOD...)
etalonnage.plot_calibration()                 # droite d'étalonnage et graphe des résidus
etalonnage.predict([0.43, 0.44], sample_name="inconnu")   # concentration d'une solution à partir de ses absorbances
```

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [D.1]</b></span></td></tr></table>
    
Les mesures réelles utilisées ici proviennent de la page [*The Beer-Lambert Law and Spectrophotometric Analysis*](https://stevesopenlab.org/the-beer-lambert-law-and-spectrophotometric-analysis/) (Steve's Open Lab, 2024), qui décrit l'expérience complète. Pour illustrer la thématique  **Données chimiques, "outliers", écart au modèle**, les données reportées dans `ML-data/Titration/Titration_KMnO4.xlsx` ont été étendues pour disposer de plus de points et adaptées.

<u>**Partie 1. Lecture des données**</u>

1. Chargez dans un DataFrame `pandas` les données du premier onglet, nommé `ext. data`, du fichier `ML-data/Titration/Titration_KMnO4.xlsx`. Affichez-le.
2. Faites l'étalonnage avec tous les points. Notez la valeur de *R*<sup>2</sup>.
3. Observez le graphe des résidus.
    - L'un des points s'écarte nettement des autres. Lequel ? S'écarte-t-il au-dessus ou au-dessous de la droite ? Proposez une erreur expérimentale qui pourrait l'expliquer.
    - Si l'on ignore ce point, les autres résidus sont-ils répartis au hasard autour de zéro ? Décrivez leur allure.
4. Lequel de ces deux défauts est un **outlier**, et lequel est un **écart au modèle** ?
5. *R*<sup>2</sup> suffisait-il à détecter les deux problèmes rencontrés ? Quel outil a été décisif ?

<u>**Partie 2. Corriger l'étalonnage**</u>

1. Retirez l'outlier du DataFrame, puis ne gardez que les étalons d'absorbance inférieure à la valeur seuil usuelle en colorimétrie. Refaites l'étalonnage.
2. Comparez la pente, l'ordonnée à l'origine *R*<sup>2</sup> et les résidus à ceux de la première partie. 
3. D'après la loi de Beer-Lambert, l'ordonnée à l'origine devrait être nulle. Est-ce le cas ici, compte tenu de son incertitude ?
4. En supposant une cuve de 1,00 cm, déduisez de la pente le coefficient d'absorption molaire *ε* du permanganate à 524 nm.
5. On a mesuré trois fois de suite l'absorbance d'une même solution inconnue, dans les mêmes conditions (mesures de *répétabilité*) : *A* = 0,512 ; 0,520 ; 0,507. Déterminez sa concentration avec la méthode `predict()`. Pourquoi a-t-on intérêt à répéter la mesure plutôt que de se contenter d'une seule valeur ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

<u>**Lire un fichier Excel.**</u> La fonction `read_excel` de `pandas` charge un onglet d'un fichier Excel dans un DataFrame :

```python
import pandas as pd
df = pd.read_excel("chemin/vers/fichier.xlsx", sheet_name="nom de l'onglet")
display(df)
```

- `sheet_name` désigne l'onglet à lire, par son nom (entre guillemets) ou par son numéro (`0` pour le premier). Sans cet argument, c'est le premier onglet qui est lu.
- La première ligne de l'onglet sert de noms de colonnes. On accède ensuite à une colonne par son nom, par exemple `df["A"]`.
- `df.columns` affiche la liste des noms de colonnes : utile pour les recopier sans faute.

<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

Dans ce notebook, vous trouverez une syntaxe du type

```python
import pandas as pd
df = pd.read_excel(get_data("chemin/vers/fichier.xlsx"), sheet_name="nom de l'onglet")
display(df)
```

<br>

`get_data(...)` est conçu pour lire des fichiers dans n'importe quel environnement de travail, y compris Google Colab. Il va chercher le fichier de données dont vous avez besoin : s’il est déjà présent localement (par exemple si vous avez cloné le dépôt pyPhysChem), il l’utilise directement ; sinon, il le télécharge automatiquement depuis GitHub. Vous n’avez rien à faire de particulier : utilisez-le simplement à la place du chemin du fichier, par exemple **`pd.read_excel(get_data("..."))`** au lieu de **`pd.read_excel("...")`**.

`get_data()` est automatiquement disponible grâce à la cellule d’initialisation en début de notebook (celle qui exécute `colab_init.py`).
</div>

<br>

<u>**Retirer une ligne d'un DataFrame.**</u> On la désigne par son numéro (son *index*), affiché à gauche du tableau quand on fait `display(df)` :

```python
dfSansOutlier = df.drop(index=i)
```

où `i` est l'index de la ligne à retirer. `df` n'est pas modifié. Le résultat (`dfSansOutlier`) est un nouveau DataFrame.

<br>

<u>**Filtrer les lignes d'un DataFrame.**</u> Pour ne garder que les lignes qui vérifient une condition, on place cette condition entre crochets :

```python
dfValide = df[df["A"] < valeur_du_seuil]
```

`df["A"] < valeur_du_seuil` produit une série de `True`/`False`, une valeur par ligne ; `df[...]` ne conserve que les lignes marquées `True`. Comme pour le filtrage, `df` n'est pas modifié

**Les deux opérations s'enchaînent** : on peut filtrer `dfSansOutlier` plutôt que `df`.

</div>

In [ ]:
# Entrez le code python



#### [D.2.2] Les vins vinho verde : la composition chimique trahit-elle la couleur ?

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Le jeu de données *Wine Quality* rassemble les analyses physico-chimiques de 6497 vins portugais *vinho verde*, rouges et blancs (Cortez *et al.*, *Decision Support Systems* **47**, 547, 2009 ; [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/wine+quality)). Pour chaque vin, on dispose de 11 grandeurs mesurées en laboratoire, d'une note de qualité attribuée par des dégustateurs, et de sa couleur.

| Colonne | Grandeur | Unité |
|---|---|---|
| `fixed acidity` | acidité fixe (acide tartrique) | g/L |
| `volatile acidity` | acidité volatile (acide acétique) | g/L |
| `citric acid` | acide citrique | g/L |
| `residual sugar` | sucres résiduels | g/L |
| `chlorides` | chlorures (NaCl) | g/L |
| `free sulfur dioxide` | SO<sub>2</sub> libre | mg/L |
| `total sulfur dioxide` | SO<sub>2</sub> total | mg/L |
| `density` | masse volumique | g/cm<sup>3</sup> |
| `pH` | pH | |
| `sulphates` | sulfates (K<sub>2</sub>SO<sub>4</sub>) | g/L |
| `alcohol` | teneur en alcool | % vol. |
| `quality` | note de qualité (dégustation, de 0 à 10) | |
| `color` | couleur du vin : `red` (rouge) ou `white` (blanc) | |

Le SO<sub>2</sub> ajouté au vin ne reste pas entièrement sous forme libre : une partie se combine à d’autres molécules du vin, d'où la quantification séparée d'une part de la forme libre, qui est la partie active qui protège le vin contre l’oxydation et les micro-organismes et d'autre part de la quantité totale de SO<sub>2</sub>. En œnologie, « sulfites » est le nom courant du SO<sub>2</sub> et de ses formes dissoutes. En solution aqueuse, et donc dans le vin, le SO<sub>2</sub> donne un équilibre acido-basique :

SO<sub>2</sub>(aq) + H<sub>2</sub>O ⇌ HSO<sub>3</sub><sup>−</sup> + H<sup>+</sup> ⇌ SO<sub>3</sub><sup>2−</sup> + 2 H<sup>+</sup>

**Question posée.** La couleur rouge d'un vin est due aux **anthocyanes**, des pigments extraits de la peau du raisin. Ces pigments n'ont pas été dosés dans cette base de données : aucune des grandeurs mesurées n'est directement liée à la couleur. Peut-on malgré tout deviner la couleur d'un vin à partir de ses seules analyses chimiques ? Autrement dit, certaines grandeurs mesurées peuvent-elles servir d'**indicateurs indirects** (on parle de ***proxys***) de la couleur ? Si oui, lesquelles ?

On se limite ici à une **analyse graphique**.
</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [D.2]</b></span></td></tr></table>

**Partie 1. Découverte des données**

1. Chargez le fichier `ML-data/winequality-RW.xlsx` dans un DataFrame `pandas`, puis affichez-le.
2. Combien de vins contient la base ? Combien de rouges, combien de blancs ? Les deux couleurs sont-elles également représentées ?
3. Comptez les lignes en double avec `df.duplicated().sum()`. Proposez une explication. Nettoyez la base de données et travaillez sur un DataFrame débarrassé de ces doublons.

**Partie 2. Comparaison des rouges et des blancs**

1. Calculez la valeur moyenne de chaque grandeur chimique pour les vins rouges et pour les vins blancs. Quelles grandeurs vous semblent les plus différentes entre les deux couleurs ?
2. Exécutez la cellule qui trace, pour chaque grandeur, les boîtes à moustaches des vins rouges et des vins blancs. Vous pouvez demander de l'aide à une IAg
    - Pour quelles grandeurs se recouvrent-elles presque entièrement ?
    - Pour quelles grandeurs les deux boîtes sont-elles nettement séparées ? Les conclusions sont-elles les mêmes qu'à la première question ?
3. Tracez la teneur en chlorures en fonction du SO<sub>2</sub> total, en colorant chaque point selon la couleur du vin. Utilisez une échelle logarithmique pour les chlorures. Les deux familles de vins se distinguent-elles ? Où se trouvent les vins « mal placés » ?

**Partie 3. Interprétation chimique**

1. Les vins blancs contiennent en moyenne trois fois plus de SO<sub>2</sub> total que les rouges. Sachant que le SO<sub>2</sub> est ajouté comme antioxydant et antiseptique, et que les vins rouges contiennent des polyphénols (tanins, anthocyanes) extraits des peaux de raisin, proposez une explication.
2. Les vins rouges ont une acidité volatile et une teneur en chlorures plus élevées. Quelle étape de la vinification, propre aux vins rouges, pourrait l'expliquer ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

La bibliothèque graphique `seaborn` est particulièrement bien adaptée pour travailler avec des données contenues dans des DataFrames et pour faire des analyses graphiques diversifiées. On l'importe habituellement sous le nom `sns` :

```python
import seaborn as sns
```

<br>

La [galerie d'exemples de seaborn](https://seaborn.pydata.org/examples/index.html) montre la plupart des types de graphiques disponibles, avec leur code.

</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

<u>**Lire une boîte à moustaches.**</u>

- le trait au milieu de la boîte est la **médiane** : la moitié des vins a une teneur en alcool inférieure, l'autre moitié une teneur supérieure ;
- la boîte va du **1<sup>er</sup> quartile** (Q<sub>1</sub>, 25 % des valeurs en dessous) au **3<sup>e</sup> quartile** (Q<sub>3</sub>, 75 % des valeurs en dessous) : elle contient la moitié centrale des données. Sa hauteur est l'**écart interquartile** IQR = Q<sub>3</sub> − Q<sub>1</sub> ;
- les **moustaches** s'arrêtent sur la dernière valeur mesurée avant la limite Q<sub>3</sub> + 1,5 × IQR (en haut) ou Q<sub>1</sub> − 1,5 × IQR (en bas) ;
- les **points isolés** au-delà des moustaches sont des **valeurs atypiques** (*outliers* au sens statistique). Elles sortent de l'ordinaire, mais ne sont pas forcément des erreurs : elles peuvent révéler un comportement physico-chimique particulier, qu'il faut alors chercher à comprendre.

<figure style="text-align: center">
  <img width="800px" src="https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/DS4B-svg/boxplot_lecture.svg">
  <figcaption><i><b>Figure.</b> Lecture d'une boîte à moustaches : teneur en alcool des 1359 vins rouges de la base (doublons retirés).</i></figcaption>
</figure>

<br>
Deux boîtes qui ne se chevauchent pas indiquent que les deux groupes sont bien séparés pour cette grandeur.

</div>

In [ ]:
# Entrez le code python



<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

Sans avoir dosé les pigments responsables de la couleur, une simple analyse graphique des données chimiques permet de distinguer très largement les vins rouges des vins blancs. Au-delà du cas du vin, cet exercice illustre plusieurs règles générales de l'analyse de données chimiques :

- **Nettoyer avant d'analyser.** Une base de données réelle n'est jamais parfaite : ici, près d'une ligne sur cinq était un doublon.
- **Comparer des distributions, pas seulement des moyennes.** Deux moyennes très différentes ne garantissent pas que les groupes se distinguent (cas des sucres résiduels) : les boîtes à moustaches montrent le recouvrement.
- **Une valeur atypique n'est pas forcément une erreur.** Elle se signale par les statistiques, mais se juge par les connaissances physico-chimiques.
- **Croiser plusieurs grandeurs.** Aucune grandeur ne suffit seule, mais la combinaison du SO<sub>2</sub> total et des chlorures sépare presque parfaitement les deux familles.
- **Corrélation n'est pas causalité.** Les grandeurs les plus discriminantes **ne sont pas la cause de la couleur**: ce sont des **proxys**, liés à la couleur par une cause commune, la méthode de vinification. Et pour certaines (les chlorures), les données seules ne suffisent pas à expliquer le lien.

Ce qu'on a fait ici "à l'œil" peut être automatisé : des méthodes d'**apprentissage automatique** (*machine learning*) savent combiner les 11 grandeurs à la fois pour prédire la couleur d'un vin, ou même sa note de qualité.

</div>
</div>

## [E] Représentations moléculaires

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Qu'est-ce qu'une molécule ?**</u></span>

Une molécule est un ensemble d'atomes reliés par des liaisons chimiques. Le nombre d'atomes peut aller de quelques-uns (par exemple dans les médicaments de faible masse molaire) à plusieurs milliers (dans les protéines), voire plusieurs millions (dans les molécules d'ADN).

Les propriétés d'une molécule résultent, de manière non triviale, des interactions entre les atomes qui la constituent. Du point de vue de la physique, les molécules sont des objets tridimensionnels complexes, dynamiques plutôt que statiques, et qui présentent des propriétés quantiques. D'après les principes de la physique quantique, la connaissance de la fonction d'onde exacte $\Psi(\textbf{R},t)$ d'une molécule donne accès à toutes ses propriétés.

Selon le problème à résoudre, une vision simplifiée des molécules peut s'avérer utile.

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Représentations moléculaires pour les humains**</u></span>

La façon la plus simple d'identifier une molécule est son nom usuel, par exemple l'aspirine. Ce n'est possible que pour des molécules bien connues.

On peut aussi utiliser les nomenclatures IUPAC de chimie organique et inorganique. L'objectif est d'associer à chaque molécule possible un nom facile à comprendre et qui l'identifie sans ambiguïté.

*Exemple* : la molécule d'aspirine a pour nom IUPAC acide 2-acétyloxybenzoïque. À partir de ce nom, il est possible de dessiner la représentation 2D de l'aspirine. En pratique, les chimistes raisonnent le plus souvent sur des **structures chimiques en 2D**, alors que **les molécules existent en réalité dans l'espace physique à 3D**. Les représentations 2D sont conçues pour reproduire l'habitude de dessiner des formules développées sur papier.


<p style="text-align:center;"><img src="https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/Molecules-data/Aspirin.png" alt="aspirine" width="15%"/></p>

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Représentations moléculaires à l'ère de la chémoinformatique**</u></span>

Pour que les chimistes puissent tirer parti de la chémoinformatique, il est important de comprendre comment les logiciels de chimie et les bibliothèques informatiques courantes stockent et analysent les structures chimiques. La chémoinformatique repose donc sur le **codage et l'utilisation de représentations des structures moléculaires et des données associées**, compréhensibles à la fois par les scientifiques et par les algorithmes, y compris les **algorithmes d'apprentissage automatique** (*machine learning*).

La conformation 3D d'une molécule et la position relative de ses groupes fonctionnels sont cependant souvent déterminantes, car elles conditionnent ses propriétés. Les coordonnées atomiques peuvent être déterminées expérimentalement (typiquement par diffraction des rayons X) ou calculées (par champs de forces, chimie quantique, dynamique moléculaire ou modèles composites comme le *docking*).

---

Le **SMILES** (*Simplified Molecular Input Line Entry System*) est la représentation textuelle la plus utilisée. Elle est prise en charge par tous les outils courants, et constitue donc un bon point de départ. Le SMILES combine des lettres, des chiffres et des symboles pour représenter les atomes et les liaisons d'une molécule : les lettres identifient les éléments, les symboles précisent la connectivité et les ramifications. Le SMILES de l'aspirine est `CC(=O)OC1=CC=CC=C1C(=O)O`, où `C1=CC=CC=C1` représente le cycle aromatique, tandis que `CC(=O)O` et `C(=O)O` correspondent respectivement aux groupes fonctionnels acétyloxy et acide carboxylique.

</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

**<span style="color:red">Prérequis.</span> Connaissances de base sur :**
- les bases de données chimiques, comme [PubChem](https://pubchem.ncbi.nlm.nih.gov/) ou [ChemSpider](https://www.chemspider.com/) ;
- le SMILES (*Simplified Molecular Input Line Entry System*), une notation linéaire qui décrit la structure des espèces chimiques à l'aide de courtes chaînes de caractères ASCII ([Wikipédia](https://fr.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_Specification)).

</div>

### [E.1] Le format SMILES

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Le SMILES décrit une molécule par une simple ligne de texte, en parcourant ses atomes de proche en proche. Quelques règles suffisent pour lire et écrire la plupart des molécules organiques :

- **Atomes** : chaque atome est désigné par son symbole (`C`, `N`, `O`, `Cl`…). Les hydrogènes ne sont pas écrits : ils sont ajoutés implicitement pour compléter la valence de chaque atome.
- **Liaisons** : une liaison simple n'est pas écrite ; une liaison double s'écrit `=`, une liaison triple `#`.
- **Ramifications** : une ramification s'écrit entre parenthèses.
- **Cycles** : un cycle est ouvert puis refermé par le même chiffre, placé après les deux atomes qui se rejoignent.
- **Aromaticité** : les atomes d'un cycle aromatique peuvent s'écrire en minuscules (`c`, `n`, `o`).
- **Crochets** : un atome entre crochets est décrit explicitement, ce qui permet de préciser une charge (`[NH4+]`, `[O-]`) ou un nombre d'hydrogènes inhabituel (`[CH2]`, pour un radical).

| Molécule | SMILES |
|---|---|
| éthanol | `CCO` |
| éthylène | `C=C` |
| acétylène | `C#C` |
| acide acétique | `CC(=O)O` |
| cyclohexane | `C1CCCCC1` |
| benzène | `C1=CC=CC=C1` ou `c1ccccc1` |
| ion acétate | `CC(=O)[O-]` |

Une même molécule peut s'écrire avec plusieurs SMILES différents, selon l'atome de départ et l'ordre de parcours : `OCC` et `CCO` désignent tous deux l'éthanol. Pour obtenir une écriture unique, les logiciels calculent un **SMILES canonique**, utile par exemple pour repérer des doublons dans une base de données.

</div>
<br>

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**RDKit, une boîte à outils pour la chémoinformatique**</u></span>


<figure>
<img width="150px" src="https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/ML-Figures/ImagesDivers/rdkit.png" alt="RDKit" />
</figure>

`RDKit` est une bibliothèque libre de chémoinformatique, utilisable en Python. Elle permet notamment :
- de construire une molécule à partir de son SMILES et de la dessiner ;
- de calculer des propriétés moléculaires (formule brute, masse molaire, descripteurs…) ;
- de rechercher des sous-structures (un groupe fonctionnel, par exemple) ;
- de générer des géométries 3D.

Elle lit et écrit de nombreux formats, dont le SMILES et le format MDL Molfile. Très utilisée dans la recherche et dans l'industrie (pharmacie, chimie), elle s'intègre bien avec Jupyter et `pandas`. Elle s'installe avec la commande :
```bash
pip install rdkit
```

Pour aller plus loin : le guide [Getting Started with the RDKit in Python](https://www.rdkit.org/docs/GettingStartedInPython.html) et le [RDKit Cookbook](https://www.rdkit.org/docs/Cookbook.html), qui rassemble des exemples prêts à l'emploi.


L'essentiel des fonctions de base se trouve dans le module [`rdkit.Chem`](https://www.rdkit.org/docs/source/rdkit.Chem.html#module-rdkit.Chem), que l'on importe en premier.

On verra ultérieurement que les `pyphyschemtools` proposent une interface `easy_rdkit`. 
Toutes les fonctionnalités sont décrites dans la [documentation du module `cheminformatics`, qui contient la **classe `easy_rdkit`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Cheminformatics.html).


</div>

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [E.1]</b></span></td></tr></table>

1. **Lire un SMILES.** Dessinez à la main la formule développée des molécules suivantes, puis identifiez-les : `CC(C)O`, `CC(=O)C`, `CC#N`, `O=C1CCCC1`, `Oc1ccccc1`. Affichez chacune de vos molécules avec `RDKit`, ainsi que sa formule brute. 
2. **Écrire un SMILES.** Écrivez le SMILES des molécules suivantes : but-2-ène, éthylamine, toluène, acide benzoïque. Vérifiez avec `RDKit` que vos SMILES correspondent aux molécules attendues.
3. **Plusieurs SMILES pour une même molécule.** Les SMILES `CCO`, `OCC` et `C(O)C` désignent-ils la même molécule ? Et `Cc1ccccc1` et `C1=CC=CC=C1C` ? Vérifiez-le avec rdkit.
4. **Une molécule plus complexe.** La vanilline, principal arôme de la vanille, a pour SMILES `COc1cc(C=O)ccc1O`. Affichez-la et identifiez ses groupes fonctionnels.
5. **Utiliser PubChem.** Recherchez sur [PubChem](https://pubchem.ncbi.nlm.nih.gov/) le SMILES de la vitamine D.  Affichez la molécule avec `RDKit`

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

```python
from rdkit import Chem
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

mol = Chem.MolFromSmiles("CCO")    # construit la molécule à partir du SMILES
display(mol)                       # dessine la molécule
print(CalcMolFormula(mol))         # formule brute
print(Chem.MolToSmiles(mol))       # SMILES canonique
```

Si le SMILES est incorrect, `Chem.MolFromSmiles` renvoie `None` et affiche un message d'erreur.

</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

<span style="color:red">**Les antislashs en Python**</span>

Dans une chaîne de caractères Python, l'antislash `\` a un sens particulier : il introduit un caractère spécial (`\n` pour un retour à la ligne, par exemple). Or les SMILES copiés depuis PubChem en contiennent souvent : les symboles `/` et `\` y décrivent la stéréochimie des doubles liaisons (*E*/*Z*). Python transforme alors une partie du SMILES en caractères invisibles, et `Chem.MolFromSmiles` renvoie `None`.

La solution : faire précéder la chaîne de la lettre `r` (*raw string*, chaîne brute), qui demande à Python de conserver les antislashs tels quels :

```python
vD = Chem.MolFromSmiles(r"C[C@H](CCCC(C)C)[C@H]1CC[C@@H]\2[C@@]1(CCC/C2=C\C=C/3\C[C@H](CCC3=C)O)C")
```

Par précaution, on peut écrire **tous** les SMILES sous la forme `r"..."` : cela ne change rien pour ceux qui ne contiennent pas d'antislash.

</div>

In [ ]:
# Entrez le code python



### [E.2] Représentation 3D

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Pourquoi la 3D ?**</u></span>

Le SMILES et les formules développées décrivent la **connectivité** d'une molécule : quels atomes sont liés entre eux. Mais une molécule réelle occupe l'espace, et beaucoup de ses propriétés dépendent de sa forme :
- la **stéréochimie** : deux énantiomères ont la même connectivité, mais des propriétés biologiques parfois très différentes (odeur, activité d'un médicament) ;
- les **conformations** : une même molécule peut adopter plusieurs formes par rotation autour de ses liaisons simples (chaise et bateau du cyclohexane, par exemple), plus ou moins stables ;
- les **interactions** : l'encombrement stérique, les liaisons hydrogène ou la reconnaissance d'une molécule par une enzyme dépendent de la position des atomes dans l'espace.

Les coordonnées 3D des atomes peuvent être **mesurées** (diffraction des rayons X sur un cristal) ou **calculées** (champs de forces, chimie quantique). Les bases de données comme PubChem proposent des géométries 3D pour la plupart des molécules organiques, et RDKit sait en construire une à partir d'un simple SMILES. Dans ce cas, il faut noter qu'elle n'est forcément proche de la structure 3D expérimentale. Il faudrait la rafiner avec des outils de chimie quantique. 

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Formats de fichiers 3D**</u></span>

Plusieurs formats de fichiers stockent la géométrie d'une molécule. Les plus courants sont :
- le format **XYZ** : le plus simple, une ligne par atome avec son symbole et ses trois coordonnées *x*, *y*, *z* (en Å) ;
- le format **MDL Molfile** (`.mol`, ou `.sdf` pour plusieurs molécules) : il contient les coordonnées des atomes, mais aussi la liste des liaisons et leur ordre (simple, double…) ;
- le format **PDB** : conçu pour les protéines et les acides nucléiques ;
- le format **CIF** : utilisé pour les structures cristallines.

Voyons sur un exemple à quoi ressemble un fichier Molfile : celui de l'anisole (méthoxybenzène), téléchargé depuis la [base PubChem](https://pubchem.ncbi.nlm.nih.gov/compound/Anisole). Après trois lignes d'en-tête, on trouve une ligne de comptage (nombre d'atomes et de liaisons), puis le **bloc des atomes** (coordonnées *x*, *y*, *z* et symbole de chaque atome), puis le **bloc des liaisons** (numéros des deux atomes liés et ordre de la liaison). Le fichier se termine par `M  END`.

</div>

In [ ]:
# Démo

with open(get_data("Molecules-data/anisole.mol"), "r") as f:
    anisole_mol = f.read()
print(anisole_mol)

<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Visualisation interactive**</u></span>

De nombreux logiciels permettent de visualiser des molécules en 3D. Dans ce notebook, on utilise la bibliothèque `py3Dmol`, qui intègre au notebook un visualiseur interactif [3Dmol.js](http://3dmol.org) : on peut faire tourner la molécule avec la souris, zoomer, changer de représentation.

La classe `molView` de `pyphyschemtools` simplifie son utilisation. Elle peut afficher une molécule à partir d'un fichier local (Molfile, XYZ, PDB, CIF), d'une chaîne de caractères, ou directement depuis une base de données en ligne : PubChem (avec son identifiant CID), la *Crystallography Open Database* (COD) ou la *Protein Data Bank* (PDB). La commande `help(molView)` affiche toutes les options.

Toutes les fonctionnalités sont décrites dans la [documentation du module `Chem3D`, qui contient la **classe `molView`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Chem3D.html).

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [E.2]</b></span></td></tr></table>

Exécutez la commande ci-dessous après avoir analysé le code.
</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

La commande `help(molView)` révèle la richesse de ce module. En pratique on n'utilisera ici qu'un nombre réduit d'options.
</div>

In [ ]:
# Démo

from pyphyschemtools import molView

mv = molView(get_data("Molecules-data/anisole.mol"), source="file")

help(molView)

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [E.3]</b></span></td></tr></table>

1. **Une géométrie 3D.** On donne le SMILES suivant : `C1CCCCC1`.
   a. Ce SMILES `C1CCCCC1` décrit la géométrie 3D du cyclohexane. Vrai ou Faux ?
   b. Affichez la structure 3D de ce composé à partir de son SMILES avec `molView`. Lisez attentivement l'aide de `molView` (`help(molView)`) pour savoir comment lui donner un SMILES.
   c. Le cycle est-il plan ? Comment s'appelle cette conformation ?
3. **Deux molécules qui sentent différemment.** La carvone existe sous deux formes : la (*R*)-carvone, à l'odeur de menthe verte, et la (*S*)-carvone, à l'odeur de carvi (cumin des prés).
    - Recherchez sur PubChem les CID et les SMILES de ces deux molécules. En quoi leurs SMILES diffèrent-ils ?
    - Affichez leurs structures 3D côte à côte avec `molView`. Faites tourner les deux molécules avec la souris pour les orienter de la même façon : le cycle à plat dans le plan de l'écran, le groupe C=O dans la même direction. Le groupe isopropényle (–C(CH<sub>3</sub>)=CH<sub>2</sub>) pointe-t-il alors du même côté du cycle dans les deux molécules ?
    - Peut-on passer d'une molécule à l'autre par une simple rotation ? Sinon, par quelle opération géométrique ? Quelle relation existe-t-il entre ces deux molécules ?
    - Pourquoi une représentation 2D ou un SMILES sans stéréochimie ne suffirait-il pas à les distinguer ?
    - Affichez les deux molécules en 2D avec RDKit, à partir de leur SMILES. Comment la stéréochimie apparaît-elle sur ces dessins ?
</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

Une brève documentation est [disponible en ligne](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Chem3D.html). On y explique comment on peut importer une molécule d'après son CID ou comment il est possible d'afficher deux molécules côte à côte.
</div>

In [ ]:
# Entrez le code python



### [E.3] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [F] Du SMILES à la structure de Lewis. RDKit, une bibliothèque qui « connaît » la chimie

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Un SMILES est une **représentation implicite** de la molécule : il ne décrit explicitement que les atomes lourds, leurs liaisons et l'ordre de ces liaisons. Tout le reste est sous-entendu. En particulier, les hydrogènes ne sont pas écrits : chaque atome est **implicitement saturé** par autant d'atomes d'hydrogène qu'il en faut pour compléter sa valence usuelle (4 pour le carbone, 3 pour l'azote, 2 pour l'oxygène). Ainsi, `CCO` désigne CH<sub>3</sub>–CH<sub>2</sub>–OH. Les doublets non liants et l'hybridation des atomes n'y figurent pas non plus, et seules les charges formelles s'écrivent explicitement, entre crochets. Un chimiste reconstruit pourtant sans difficulté la structure complète, en appliquant les règles qu'il connaît : valence des atomes, règle de l'octet, théorie VSEPR, conjugaison.

RDKit fait la même chose. À partir d'un simple SMILES, la bibliothèque applique ces règles chimiques pour reconstruire l'information manquante :
- elle **sature implicitement** chaque atome par des hydrogènes, en complétant sa valence ;
- elle repère les **charges formelles** et les **doublets non liants** ;
- elle détermine l'**hybridation** de chaque atome ;
- elle identifie les cycles **aromatiques** et les systèmes **conjugués**.

On va dorénavant utiliser la **classe `easy_rdkit`** de `pyphyschemtools`, qui facilite l'utilisation de RDKit, par exemple en rendant ces informations visibles directement sur le dessin de la molécule, en superposant à la formule développée les doublets non liants, les lacunes électroniques et l'état d'hybridation de chaque atome : on obtient une véritable **structure de Lewis**.

Toutes les fonctionnalités sont décrites dans la [documentation du module `cheminformatics`, qui contient la **classe `easy_rdkit`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Cheminformatics.html).

</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

<span style="color:red">**Des conventions chimiques, pas un calcul.**</span> RDKit ne résout pas l'équation de Schrödinger : il applique des règles, comme le ferait un chimiste. Ses conclusions sont donc en général justes, mais il peut arriver qu'elles s'écartent de ce que prévoit une règle simple comme la VSEPR, ou qu'elles reflètent une convention plutôt qu'une réalité physique. Dans ce cas, c'est à vous de juger, et de comprendre pourquoi.

</div>

### [F.1] Découverte de `easy_rdkit`

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [F.1]</b></span></td></tr></table>

Le solution de cet exercice est intégralement donnée ci-dessous. Essayez de comprendre en quoi on a répondu aux questions posées, et copiez/collez les bouts de code utile dans les exercices suivants.

1. Créez une instance de `easy_rdkit` à partir du SMILES `C=CCC=CC=O`, avec l'option `lang="Fr"`, puis affichez la molécule avec la méthode `show_mol()`. Nommez cette molécule.
2. Cette molécule est-elle référencée dans PubChem ? Consultez l'aide de la méthode `fetch_pubchem_data` avec `help()`, puis utilisez-la pour retrouver le CID de la molécule.
3. Affichez à nouveau la molécule, en activant cette fois les options `show_Lewis`, `show_n`, `show_hybrid` et `plot_conjugation`.
    - Que représente chacune de ces options sur le dessin ?
    - Combien chaque oxygène porte-t-il de doublets non liants ? Quelle est l'hybridation de chaque carbone ?
    - Toutes les doubles liaisons sont-elles conjuguées ? Justifiez à partir de la structure de la molécule.
4. Une instance de `easy_rdkit` peut aussi être créée directement à partir du CID d'une molécule, avec `easy_rdkit.from_cid()`. Faites-le avec le CID 240. De quelle molécule s'agit-il ? Quelle partie de la molécule est conjuguée ?
5. Elle peut également être créée à partir du nom d'une molécule, avec `easy_rdkit.from_name()`. Faites-le pour l'aspirine, et affichez son nom IUPAC et son SMILES canonique.

</div>

In [ ]:
# Démo

import pyphyschemtools as t4pPC
from pyphyschemtools import easy_rdkit

#################################################################################
t4pPC.centerTitle("Instanciation de easy_rdkit avec un SMILES, et affichage basique")
mol = easy_rdkit("C=CCC=CC=O", lang="Fr")
mol.show_mol()

#################################################################################
t4pPC.centerTitle("La molécule est-celle référencée dans PubChem, et quel est son CID?")
help(easy_rdkit.fetch_pubchem_data)
mol.fetch_pubchem_data()
print()
print("CID de C=CCC=CC=O = ",mol.cid)

#################################################################################
t4pPC.centerTitle("Analyse de la structure de Lewis de la molécule")
mol.show_mol(show_Lewis=True, show_n=True, show_hybrid=True, plot_conjugation=True, size=(600,400))

#################################################################################
t4pPC.centerTitle("Analyse de la structure de Lewis de la molécule")

#################################################################################
t4pPC.centerTitle("Instanciation de easy_rdkit avec un CID de PubChem")
mol = easy_rdkit.from_cid(240, lang="Fr")
print(f"compound {mol.cid} = {mol.iupac_name}\nRDKit-canonicalized SMILES = {mol.smiles}")
mol.show_mol(show_Lewis=True, show_n=True, show_hybrid=True, plot_conjugation=True, size=(600,400))

#################################################################################
t4pPC.centerTitle("Instanciation de easy_rdkit avec un nom")
mol = easy_rdkit.from_name("aspirine", lang="Fr")
print(f"compound {mol.cid} = {mol.iupac_name}\nRDKit-canonicalized SMILES = {mol.smiles}")
mol.show_mol(show_Lewis=True, show_n=True, show_hybrid=True, plot_conjugation=True, size=(600,400))

### [F.2] Petits composés simples

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [F.2]</b></span></td></tr></table>

On donne les SMILES des molécules et ions suivants :

| Espèce | SMILES |
|---|---|
| méthane | `C` |
| ammoniac | `N` |
| ion ammonium | `[NH4+]` |
| phosphine | `P` |
| ion oxonium | `[OH3+]` |
| ion hydroxyde | `[OH-]` |
| éthanol | `CCO` |
| pentachlorure de phosphore | `ClP(Cl)(Cl)(Cl)Cl` |
| ion hexafluorophosphate | `F[P-](F)(F)(F)(F)F` |
| monoxyde de carbone | `C#O` |

Les neuf premiers SMILES sont regroupés dans une liste Python :

```python
liste_smiles = ["C", "N", "[NH4+]", "P", "[OH3+]", "[OH-]", "CCO", "ClP(Cl)(Cl)(Cl)Cl", "F[P-](F)(F)(F)(F)F"]
```

1. Pour chacune des neuf premières espèces, dessinez d'abord à la main sa structure de Lewis. Créez ensuite une instance de `easy_rdkit` et affichez sa structure de Lewis avec les options `show_Lewis`, `show_n` et `show_hybrid`. Vos structures sont-elles identiques ?
2. Comparez l'ammoniac et l'ion ammonium : combien de doublets non liants porte l'azote dans chaque cas ? Son hybridation change-t-elle ? Où apparaît la charge positive ?
3. Faites la même comparaison entre l'ion oxonium et l'ion hydroxyde : nombre de doublets non liants de l'oxygène, charge formelle.
4. Comparez l'ammoniac et la phosphine. 
5. Dans PCl<sub>5</sub> et PF<sub>6</sub><sup>−</sup>, combien d'électrons entourent le phosphore ? La règle de l'octet est-elle respectée ? RDKit attribue au phosphore une hybridation : discutez avec votre enseignant de la pertinence de la notion d'hybridation pour ces éléments.
6. Faites de même pour le monoxyde de carbone, avec le SMILES `C#O`. Que se passe-t-il ? Lisez attentivement le message d'erreur : qu'est-ce qui ne va pas ? Proposez une structure de Lewis correcte pour CO, et le SMILES correspondant. Vérifiez-le avec `easy_rdkit`.

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

- Dans un SMILES, un atome entre crochets est décrit explicitement, avec ses hydrogènes et sa charge formelle : `[NH4+]` désigne un azote portant quatre hydrogènes et une charge positive, `[O+]` serait un oxygène sans hydrogène portant une charge positive, et non pas H<sub>3</sub>O<sup>+</sup>, qui s'écrit `[OH3+]`.

- Pour afficher toutes les structures de la liste, utilisez une boucle `for` :

    ```python
    for smi in liste_smiles:
        mol = easy_rdkit(smi, lang="Fr")
        mol.show_mol(show_Lewis=True, show_n=True, show_hybrid=True)
    ```

</div>

In [ ]:
# Entrez le code python



In [ ]:
################################################################## version corrigée du smiles de CO
# Entrez le code python



### [F.3] Aromaticité, doublets non liants et hétérocycles

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Un cycle est **aromatique** s'il est plan, entièrement conjugué, et s'il compte 4*m* + 2 électrons π (règle de Hückel). Le benzène, avec ses 6 électrons π, en est l'exemple type.

Mais les électrons π ne proviennent pas forcément de doubles liaisons : un **doublet non liant** peut lui aussi rejoindre le système π, s'il occupe une orbitale 2p<sub>z</sub> perpendiculaire au cycle. Inversement, un doublet situé dans le plan du cycle ne participe pas à l'aromaticité. Savoir où se trouve un doublet non liant permet ainsi de prévoir l'aromaticité d'un cycle, mais aussi certaines de ses propriétés chimiques, comme sa basicité.

On utilise ici `easy_rdkit` pour étudier quatre cycles à cinq ou six atomes : le cyclopentadiène, l'anion cyclopentadiényle, le pyrrole et la pyridine.

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [F.3]</b></span></td></tr></table>

Pour chaque molécule ci-dessous, créez une instance de `easy_rdkit` à partir du SMILES donné, puis :
- affichez sa structure de Lewis avec la méthode `show_mol()`, en activant les options `show_Lewis`, `show_n`, `show_hybrid` et `plot_conjugation`.
- faites un deuxième affichage avec uniquement l'option `plot_aromatic=True`

1. **Le cyclopentadiène et l'anion cyclopentadiényle.**
    - Le cyclopenta-1,3-diène, C<sub>5</sub>H<sub>6</sub> (SMILES : `C1=CC=CC1`), est-il aromatique ? Quelle est l'hybridation de chaque carbone ?
    - L'anion cyclopentadiényle, C<sub>5</sub>H<sub>5</sub><sup>−</sup> (SMILES : `C1=CC=C[CH-]1`), est obtenu en arrachant un proton H<sup>+</sup> au CH<sub>2</sub> du cyclopentadiène. Que devient ce carbone ? Le cycle est-il aromatique ? Combien d'électrons π compte-t-il ? Est-ce cohérent avec la règle de Hückel ?
2. **Le pyrrole**, C<sub>4</sub>H<sub>5</sub>N (SMILES : `C1=CC=CN1`), est un cycle aromatique à cinq atomes contenant un groupe NH.
    - Quelle est l'hybridation de l'azote ? Est-ce celle que prévoit la théorie VSEPR (l'azote est de type AX<sub>3</sub>E<sub>1</sub>) ?
    - Combien d'électrons π compte le cycle ? D'où vient chacun d'eux ?
3. **La pyridine**, C<sub>5</sub>H<sub>5</sub>N (SMILES : `C1=CC=NC=C1`), est un cycle aromatique à six atomes, dans lequel un groupe CH du benzène est remplacé par un azote.
    - Combien d'électrons π compte le cycle ?
    - Le doublet non liant de l'azote participe-t-il au système π ? Comparez au pyrrole.
4. Sachant qu'une base doit pouvoir fixer un proton H<sup>+</sup> sur un doublet non liant, laquelle de ces deux molécules, pyrrole ou pyridine, est la plus basique ? Justifiez.
5. **Conjugaison ou aromaticité ?** Comparez, pour les quatre molécules, les zones mises en évidence par `plot_conjugation` et par `plot_aromatic`. Une molécule conjuguée est-elle forcément aromatique ? Quelle condition supplémentaire faut-il ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

Dans un SMILES, un atome portant une charge s'écrit entre crochets, avec ses hydrogènes et sa charge : `[CH-]` désigne un carbone portant un hydrogène et une charge négative.

</div>

In [ ]:
# Entrez le code python



### [F.4] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [G] Symétrie moléculaire

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**La symétrie en chimie**</u></span>

La symétrie moléculaire décrit les symétries d'une molécule et permet de classer les molécules selon leur symétrie. C'est une notion fondamentale en chimie, car elle permet de prévoir ou d'expliquer de nombreuses propriétés :
- les transitions spectroscopiques permises (UV-visible, infrarouge, Raman) ;
- la présence ou l'absence d'un moment dipolaire ;
- la chiralité d'une molécule ;
- la construction des orbitales moléculaires : méthode de Hückel, théorie du champ des ligands en chimie de coordination, règles de Woodward–Hoffmann.

Beaucoup de molécules sont symétriques. Si on fait tourner la molécule d'eau de 180° autour de l'axe qui passe par l'oxygène et coupe l'angle H–O–H en deux, les deux hydrogènes échangent leurs places : la molécule obtenue est indiscernable de la molécule de départ. On dit que cette rotation est une **opération de symétrie** de la molécule.

Il existe cinq types d'opérations de symétrie, chacune associée à un **élément de symétrie** :

| Opération | Notation | Élément de symétrie |
|---|---|---|
| identité | E | — |
| rotation de 360°/n | C<sub>n</sub> | axe de symétrie |
| réflexion | σ | plan de symétrie |
| inversion | i | centre d'inversion |
| rotation-réflexion | S<sub>n</sub> | axe impropre |

L'ensemble des opérations de symétrie d'une molécule forme un **groupe ponctuel**, noté en notation de Schoenflies : C<sub>2v</sub> pour l'eau, D<sub>6h</sub> pour le benzène... L'outil mathématique sous-jacent est la **théorie des groupes** : chaque groupe a une **table de caractères**, qui décrit ses **représentations irréductibles**, c'est-à-dire les différentes façons dont un objet attaché à la molécule (orbitale, vibration...) peut se transformer par les opérations de symétrie. La même démarche est utilisée en cristallographie pour décrire la symétrie des cristaux, avec une autre notation (Hermann-Mauguin).

Pour en savoir plus : [Symétrie moléculaire (Wikipédia)](https://fr.wikipedia.org/wiki/Sym%C3%A9trie_mol%C3%A9culaire).

<span style="color:blue"><u>**Le module `symmetry` de `pyphyschemtools`**</u></span>

Une opération de symétrie transforme les coordonnées des atomes : c'est une matrice 3×3. Python applique cette matrice et repère où va chaque atome : c'est le **suivi des atomes**. Tout le reste en découle. Le module `symmetry` de `pyphyschemtools` fait ces opérations mathématiques pour vous, y compris l'exploitation des tables de caractères, ce qui permet de se concentrer sur la chimie. Il permet plus précisément :

1. d'écrire les opérations de symétrie sous forme de matrices 3×3, et de les combiner ;
2. d'afficher la table de caractères de n'importe quel groupe ponctuel courant (55 groupes) ;
3. de trouver le groupe ponctuel d'une molécule à partir de sa structure 3D ;
4. de visualiser en 3D les éléments de symétrie, et la molécule avant et après une opération ;
5. de suivre la façon dont les atomes s'échangent sous l'effet des opérations de symétrie ;
6. de déterminer la symétrie des orbitales atomiques et moléculaires, ainsi que celle des modes de vibrations, et en déduire le nombre de bandes actives en infrarouge et en Raman.

Une molécule est représentée par un objet `MolSym`, construit à partir de sa **structure 3D**, c'est-à-dire de la position de chaque atome dans l'espace. <span style="color:red">`MolSym` ne lit pas lui-même les structures : il sous-traite cette tâche à `molView` (module `Chem3D`), qui sait lire un SMILES, un fichier xyz ou une structure de la base PubChem, et construire la géométrie 3D si nécessaire.</span> `MolSym` détermine ensuite le groupe ponctuel de la molécule, et l'enregistre dans **l'attribut `group`**.

Toutes les fonctionnalités sont décrites dans la [documentation du module `symmetry`, qui contient la **classe `MolSym`, ainsi que de nombreuses fonctions**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Symmetry.html).

</div>

### [G.1] Exercice démo 1 : la symétrie de la molécule d'eau

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [G.1]</b></span></td></tr></table>

L'exécution de la cellule Python suivante répond aux questions ci-dessous. Analysez le code ainsi que les résultats et faites le lien avec les questions. Soyez prêt(e)s à faire les copier-coller nécessaires dans les exercices qui suivront.

1. **Sur papier** : quels sont les éléments de symétrie de la molécule d'eau ? En déduire son groupe ponctuel de symétrie.
2. Récupérez avec `molView` la structure 3D de l'eau dans la base de données PubChem, sachant que son **CID est 962**.
3. À l'aide de la classe `MolSym`, déterminez le **groupe ponctuel de symétrie** de l'eau.
4. À l'aide de la méthode `orientation()`, affichez **l'orientation standard** de la molécule dans le repère (x, y, z) : dans quel plan se trouve-t-elle ? Selon quel axe est orienté l'axe principal ?
5. Affichez la table de caractères du groupe avec la méthode `character_table()`. Les en-têtes des colonnes donnent la liste des opérations de symétrie : combien y en a-t-il ?
6. Visualisez les **éléments de symétrie en 3D** avec la méthode `show_elements()` (la vue peut être tournée à la souris), d'abord tous ensemble, puis un par un. Décrivez chaque élément par rapport à la molécule.
7. Affichez la représentation matricielle des opérations de symétrie. 
8. **Comparez** avec votre réponse à la question 1.

</div>

In [ ]:
# Démo

from pyphyschemtools import molView
from pyphyschemtools import MolSym
from IPython.display import Markdown, display

#################################################################################
t4pPC.centerTitle("2. Récupération de la structure 3D (PubChem)")
mv_eau = molView(962, source="cid")

#################################################################################
t4pPC.centerTitle("3. Groupe ponctuel")
eau = MolSym.from_molView(mv_eau, lang="Fr")
print("GPS de H2O = ", eau.group)

#################################################################################
t4pPC.centerTitle("4. Orientation")
eau.orientation()

#################################################################################
t4pPC.centerTitle("5. Table de caractères")
display(eau.character_table())
display(Markdown(r"""
Le groupe C<sub>2v</sub> contient **4 opérations** : E, C<sub>2</sub>, σ<sub>v</sub>(xz) et σ<sub>v</sub>'(yz).
"""))

#################################################################################
t4pPC.centerTitle("6. Éléments de symétrie")
eau.show_elements(labels=True)
eau.show_elements("C2")
eau.show_elements("σv(xz)")
eau.show_elements("σv'(yz)")
display(Markdown(r"""
- **C<sub>2</sub>** (en rouge) : axe passant par l'oxygène, bissectrice de l'angle H–O–H. Une rotation de 180° échange les deux hydrogènes.
- **σ<sub>v</sub>(xz)** : plan perpendiculaire au plan de la molécule, contenant l'axe C<sub>2</sub>. La réflexion échange les deux hydrogènes.
- **σ<sub>v</sub>'(yz)** : plan de la molécule. La réflexion laisse tous les atomes en place.
- **E** (identité) : ne change rien ; il n'y a pas d'élément à dessiner.
"""))

#################################################################################
t4pPC.centerTitle("7. Représentation matricielle des opérations de symétrie")
eau.matrices()
display(Markdown(r"""
Chaque opération de symétrie s'écrit comme une **matrice 3×3**. Pour obtenir les nouvelles coordonnées (x', y', z') d'un atome après l'opération, on multiplie cette matrice par ses coordonnées (x, y, z).
Par exemple, C<sub>2</sub>(z) change x en −x et y en −y, et laisse z inchangé.

Toute la machinerie du module `symmetry` repose sur ces matrices : il les applique aux coordonnées des atomes, puis regarde sur quel atome chacun se retrouve après application d'une OS.
C'est ainsi qu'il vérifie que la molécule est inchangée, qu'il suit l'échange des atomes, et qu'il calcule les caractères.
"""))

#################################################################################
t4pPC.centerTitle("1. et 8. Bilan")
display(Markdown(r"""
L'eau possède un axe C<sub>2</sub> et deux plans de symétrie qui contiennent cet axe : le plan de la molécule et le plan perpendiculaire. Avec l'identité E, cela fait 4 opérations de symétrie :
c'est le groupe **C<sub>2v</sub>** (un axe C<sub>2</sub>, des plans **v**erticaux, qui contiennent l'axe). L'eau ne possède pas de centre d'inversion.
"""))

### [G.2] Exercice démo 2 : les caractères dans la base des OA s de valence de l'eau

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [G.2]</b></span></td></tr></table>

On considère la base formée des trois orbitales atomiques s de valence de l'eau : 2s de O0, 1s de H1 et 1s de H2. Chaque opération de symétrie transforme cette base en elle-même. L'exécution de la cellule Python suivante répond aux questions ci-dessous.

1. Avec la méthode `track()`, indiquez sur quel atome arrive chaque atome pour chacune des opérations E, C<sub>2</sub>, σ<sub>v</sub>(xz) et σ<sub>v</sub>'(yz).
2. Avec la méthode `show_operation()` et l'option `highlight=1`, suivez l'atome H1 au cours des opérations C<sub>2</sub> et σ<sub>v</sub>'(yz). Où arrive-t-il ?
3. Une orbitale s est sphérique : elle est seulement déplacée avec son atome. Avec la méthode `representation_matrix()`, affichez la matrice de chaque opération dans la base (s(O0), s(H1), s(H2)). Comment la construire à partir des résultats de la question 1 ?
4. Le **caractère** χ d'une opération est la trace de sa matrice, c'est-à-dire la somme des éléments diagonaux. Montrez qu'il est égal au nombre d'atomes qui restent en place. Calculez χ pour les quatre opérations.
5. Vérifiez vos caractères avec la méthode `gamma_orbitals("s")`. On note Γ<sub>s</sub> la représentation obtenue.
6. `gamma_orbitals()` réduit aussi Γ<sub>s</sub> en représentations irréductibles. Interprétez le résultat : quelle est la symétrie de l'OA 2s de l'oxygène ? Quelles combinaisons des deux OA 1s des hydrogènes peut-on former ?

</div>

In [ ]:
# Démo

#################################################################################
t4pPC.centerTitle("1. Suivi des atomes")
eau.track("E")
eau.track("C2")
eau.track("σv(xz)")
eau.track("σv'(yz)")

#################################################################################
t4pPC.centerTitle("2. Que devient l'atome H1 ?")
eau.show_operation("C2", highlight=1)
eau.show_operation("σv'(yz)", highlight=1)
display(Markdown(r"""
Par C<sub>2</sub> (comme par σ<sub>v</sub>(xz)), H1 arrive à la place de H2 : les deux hydrogènes sont échangés. Par σ<sub>v</sub>'(yz), qui est le plan de la molécule, H1 reste en place.
"""))

#################################################################################
t4pPC.centerTitle("3. Matrices dans la base des OA s")
eau.representation_matrix("E")
eau.representation_matrix("C2")
eau.representation_matrix("σv(xz)")
eau.representation_matrix("σv'(yz)")
display(Markdown(r"""
Dans la colonne de chaque orbitale, on met un 1 sur la ligne de l'orbitale sur laquelle elle arrive, et des 0 partout ailleurs. Par exemple, C<sub>2</sub> envoie s(H1) sur s(H2) : la colonne de s(H1) contient un 1 sur la ligne de s(H2). On obtient une **matrice de permutation**.
"""))

#################################################################################
t4pPC.centerTitle("4. Caractères")
display(Markdown(r"""
Un 1 sur la diagonale correspond à une orbitale qui revient sur elle-même, donc à un atome qui reste en place. La trace est donc le nombre d'atomes immobiles :

| | E | C<sub>2</sub> | σ<sub>v</sub>(xz) | σ<sub>v</sub>'(yz) |
|---|---|---|---|---|
| Γ<sub>s</sub> | 3 | 1 | 1 | 3 |
"""))

#################################################################################
t4pPC.centerTitle("5. et 6. Vérification et réduction")
eau.gamma_orbitals("s")
display(Markdown(r"""
On retrouve les caractères (3, 1, 1, 3). La réduction donne, par ensemble d'atomes équivalents :
- **O 2s → A<sub>1</sub>** : l'OA de l'oxygène, placée sur tous les éléments de symétrie, est totalement symétrique ;
- **H 1s → A<sub>1</sub> + B<sub>2</sub>** : les deux OA 1s se combinent en une combinaison en phase, s(H1) + s(H2), de symétrie A<sub>1</sub>, et une combinaison en opposition de phase, s(H1) − s(H2), de symétrie B<sub>2</sub>.

Au total, Γ<sub>s</sub> = 2A<sub>1</sub> + B<sub>2</sub>. Seules des orbitales de même symétrie peuvent se combiner pour former des orbitales moléculaires : c'est la base des diagrammes d'OM.
"""))

### [G.3] Exercice : la symétrie des OA de valence de l'eau

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [G.3]</b></span></td></tr></table>

On considère la base de **toutes les orbitales atomiques de valence** de l'eau.

1. Quelles sont les OA de valence de l'oxygène et des hydrogènes ? Combien la base contient-elle d'orbitales ?
2. Pour chacune des opérations E, C<sub>2</sub>, σ<sub>v</sub>(xz) et σ<sub>v</sub>'(yz), indiquez ce que devient chaque OA de la base. Utilisez `track()` pour savoir où vont les atomes, si besoin. Pour les OA 2p de l'oxygène, réfléchissez à l'éventuel échange des lobes + et − de l'orbitale, c'est-à-dire un changement de signe.
3. Déduisez-en le caractère de chaque opération, puis remplissez la ligne Γ<sub>val</sub> sous les colonnes de la table de caractères.
4. Vérifiez vos caractères avec la méthode `gamma_orbitals()`, et relevez la réduction de Γ<sub>val</sub> en représentations irréductibles.
5. Seules des OA de même symétrie peuvent se combiner pour former des orbitales moléculaires. Quelles OA peuvent se combiner ? Quelle OA de l'oxygène reste seule ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

**Calcul d'un caractère : seules les orbitales qui restent sur place comptent.** Pour une opération R, on regarde ce que devient chaque orbitale u<sub>i</sub> de la base :
- si R u<sub>i</sub> = +u<sub>i</sub> (l'orbitale est inchangée), elle compte pour **+1** ;
- si R u<sub>i</sub> = −u<sub>i</sub> (l'orbitale change de signe), elle compte pour **−1** ;
- si R u<sub>i</sub> = u<sub>j</sub>, une autre orbitale de la base (par exemple l'orbitale similaire d'un autre atome, comme 1s(H1) → 1s(H2)), elle compte pour **0**.

Le caractère χ(R) est la somme de ces contributions.

</div>

In [ ]:
# Entrez le code python



### [G.4] Déterminer le groupe ponctuel de symétrie d'une molécule

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [G.4]</b></span></td></tr></table>

On veut déterminer le GPS des composés de la figure ci-dessous.

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/rpoteau/pyPhysChem/main/Molecules-data/GPS_planche_enonce-C.png" width="800">
</div>


On procède en trois étapes :

- on **observe** la molécule en 3D, ainsi que sa représentation de Cram sur la figure ci-dessus ;
- on **cherche ses éléments de symétrie**, et on en déduit son groupe ponctuel avec l'organigramme ;
- on **vérifie** avec le module `symmetry`.

Les molécules sont données :

- **A.** par leur SMILES (dont l'acide trichloroisocyanurique et l'ion dichloroisocyanurate) ;
- **B.** par un fichier xyz, situé dans le dossier `Molecules-data`.

```python
liste_smiles = [r"F/C=C\F", r"F/C=C/F", "Cl[Al](Cl)Cl", "NCl",
                "O=C1N(Cl)C(=O)N(Cl)C(=O)N1Cl", "O=C1N(Cl)C(=O)[N-]C(=O)N1Cl"]

liste_xyz = ["PF5_model.xyz", "SF5Cl_model.xyz", "trans-SF4Cl2_model.xyz", "cis-SF4Cl2_model.xyz",
             "mer-SF3Cl3_model.xyz", "fac-SF3Cl3_model.xyz", "BOH3_model.xyz"]
```

1. Afficher l'organigramme de détermination d'un groupe ponctuel ([voir la documentation en ligne](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Symmetry.html)).
2. Afficher en 3D toutes les molécules A et B. Faire tourner chaque molécule, et noter dans le tableau ci-dessous :
    - si elle est plane ;
    - son axe C<sub>n</sub> d'ordre le plus élevé ;
    - s'il existe des axes C<sub>2</sub> perpendiculaires à cet axe, un plan σ<sub>h</sub>, des plans σ<sub>v</sub>, un centre d'inversion.

    En déduire son groupe ponctuel à l'aide de l'organigramme.
3. Vérifier vos réponses avec `MolSym`.
4. Pour chaque molécule où votre réponse est fausse, afficher ses éléments de symétrie et le chemin suivi dans l'organigramme.
5. Discussion :
    - a. Les isomères Z et E ont-ils le même groupe ponctuel ? Et les isomères cis et trans, mer et fac ?
    - b. Le groupe ponctuel suffit-il toujours à distinguer deux isomères ?
    - c. B(OH)<sub>3</sub> est plan et possède un axe C<sub>3</sub>. Pourquoi n'est-il pas D<sub>3h</sub> ?

</div>
<br>
<div class="rq">

<table width="100%" cellpadding="6"><tr><td bgcolor="#fde9e9"><span style="color:#B71C1C">⚠️ <b>Remarque</b></span></td></tr></table>

Dans un SMILES, les symboles `/` et `\` indiquent la direction de chaque liaison simple quand on lit le SMILES de gauche à droite : `/` = « on monte », `\` = « on descend ». Ils permettent de coder les isomères Z et E.

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

- Grille de molécules 3D : `molView.view_grid(liste, n_cols=3, source="smiles")`, ou `source="file"` pour une liste de fichiers xyz.
- Utile pour la partie "observation" : ajouter un chemin devant une liste de fichiers xyz:

    ```python
    fichiers = []
    for f in liste_xyz:
        fichiers.append(get_data("Molecules-data/" + f)).
    ```

- Vérification :
    - avec une boucle sur la liste des SMILES :

        ```python
        for smi in liste_smiles:
            mol = MolSym.from_molView(molView(smi, source="smiles", display_now=False), lang="Fr", verbose=False)
            print(smi, ":", mol.group)
        ```

    - avec une boucle similaire pour les fichiers xyz, mais avec : `MolSym.from_molView(molView(get_data("Molecules-data/" + fichier), source="file", display_now=False), lang="Fr", verbose=False)`
- Éléments de symétrie : `mol.show_elements()`. Chemin dans l'organigramme : `mol.flowchart()`.

</div>

In [ ]:
# Entrez le code python



### [G.5] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [H] Chimie organique orbitalaire : molécules à liaison(s) π. Conjugaison

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Pourquoi s'intéresser aux liaisons π ?**</u></span>

Les molécules qui possèdent plusieurs liaisons doubles en alternance avec des liaisons simples, dites **conjuguées**, sont partout en chimie organique et dans le vivant :
- la **couleur** : le β-carotène, qui donne sa couleur orange à la carotte, possède onze doubles liaisons conjuguées ; la chlorophylle et la plupart des colorants alimentaires sont aussi des systèmes conjugués ;
- la **vision** : le rétinal, une molécule conjuguée de nos yeux, change de forme en absorbant la lumière, ce qui déclenche l'influx nerveux ;
- la **stabilité** : le benzène est beaucoup plus stable, et beaucoup moins réactif, que ne le laisserait prévoir sa formule de Lewis à trois doubles liaisons ;
- la **réactivité** : réactions d'addition sur les doubles liaisons, réaction de Diels-Alder, substitution électrophile aromatique…

La formule de Lewis montre vite ses limites face à ces molécules. Dans le benzène, les six liaisons C–C ont exactement la même longueur, intermédiaire entre une simple et une double liaison. Dans le butadiène, H<sub>2</sub>C=CH–CH=CH<sub>2</sub>, la liaison centrale est plus courte qu'une simple liaison habituelle. La mésomérie apporte une première explication, en dessinant plusieurs formules limites. Les **orbitales moléculaires &pi;** permettent d'aller plus loin, et de calculer ces effets : c'est l'objet de cette partie.

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**Séparation σ/π**</u></span>

Dans une molécule plane comme l'éthylène, les orbitales atomiques se répartissent en deux familles qui n'interagissent pas entre elles :
- les orbitales situées dans le plan de la molécule (2s, 2p<sub>x</sub>, 2p<sub>y</sub> des carbones, 1s des hydrogènes) forment le **squelette σ** ;
- les orbitales 2p<sub>z</sub>, perpendiculaires au plan, forment le **système π**.

Les électrons π sont les plus hauts en énergie et les plus mobiles : ce sont eux qui gouvernent la réactivité, la couleur et la conjugaison des molécules organiques. On peut donc, en première approximation, étudier le système π seul.

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**La méthode de Hückel**</u></span>

Proposée par Erich Hückel en 1931, cette méthode décrit les OM π comme des combinaisons linéaires des orbitales 2p<sub>z</sub> des atomes du système π, au prix d'approximations très simples :
- le recouvrement entre orbitales 2p<sub>z</sub> de deux atomes différents est négligé ;
- l'énergie d'un électron dans une orbitale 2p<sub>z</sub> isolée est notée $\alpha$ (**intégrale coulombienne**) ;
- l'interaction entre les orbitales 2p<sub>z</sub> de deux atomes **liés** est notée $\beta$ (**intégrale de résonance**), et elle est nulle entre atomes non liés.

$\alpha$ et $\beta$ sont deux énergies négatives. L'hamiltonien de Hückel est une matrice qui porte $\alpha$ sur la diagonale et $\beta$ entre atomes liés. Ses **valeurs propres** sont les énergies des OM π, de la forme $\varepsilon = \alpha + x\beta$, et ses **vecteurs propres** donnent les coefficients des orbitales 2p<sub>z</sub> dans chaque OM.

Malgré sa simplicité, la méthode de Hückel rend compte de nombreuses propriétés : stabilisation par conjugaison, aromaticité, longueurs de liaison, sites réactifs.

</div>
<br>
<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

<span style="color:blue"><u>**La classe `HMO` de `pyphyschemtools`**</u></span>

La classe `HMO` (*Hückel Molecular Orbitals*) applique la méthode de Hückel à partir du seul SMILES d'une molécule : elle repère les atomes du système π, construit l'hamiltonien, le diagonalise, puis remplit les OM avec les électrons π.

Deux conventions sont à retenir :
- les atomes sont numérotés **à partir de 0**, dans l'ordre du SMILES (C0, C1…) ;
- les énergies sont données en **unités de $|\beta|$**, avec la convention $\alpha = 0$.

On commence par la molécule la plus simple : l'éthylène, qui servira de référence pour toute la suite. On retiendra en particulier deux valeurs : l'énergie π de la double liaison, $E_\pi = 2\alpha + 2\beta$, et son indice de liaison π, égal à 1.

Toutes les fonctionnalités sont décrites dans la [documentation du module `huckel`, qui contient la **classe `HMO`**](https://pyphyschemtools.readthedocs.io/en/latest/theoryDocs/Huckel.html).

</div>

### [H.1] Éthène

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.1]</b></span></td></tr></table>

Exécutez la cellule ci-dessous, qui applique la méthode de Hückel à l'éthylène, puis répondez aux questions en analysant chaque sortie.

1. **Représentation 2D.** Quels atomes font partie du système π ? Quelle est leur hybridation ? Comment sont-ils numérotés ?
2. **Matrice de Hückel.** Que représentent les termes $\alpha$ et $\beta$ ? Quel est leur signe ? Pourquoi la matrice est-elle de taille 2 × 2 ?
3. **Énergies.** Combien d'électrons π compte l'éthylène ? Combien d'OM π ? Quelles sont leurs énergies, en fonction de $\alpha$ et $\beta$ ? Retrouvez la valeur de l'énergie π totale affichée par `summary()`.
4. **Diagramme d'OM.** Quelle OM est occupée ? Laquelle est vide ?Laquelle est la plus basse en énergie ? Identifiez la HO et la BV.
5. **Orbitales frontières.** Le diagramme avec l'option `"FMO"` représente aussi la HO et la BV, vues de dessus. Que représentent les couleurs des lobes ? Laquelle de ces OM est liante, laquelle est antiliante ?
6. **Population π.** Le dernier graphe indique la charge π de chaque atome et l'indice de liaison π. Quelles sont leurs valeurs ? Étaient-elles prévisibles ?

</div>

In [ ]:
# Démo

import pyphyschemtools as t4pPC
from pyphyschemtools import HMO

ethene = HMO('C=C')

t4pPC.centerTitle("2D representation")
ethene.show(show_hybridization=True)

t4pPC.centerTitle("Hückel matrix - symbolic")
ethene.huckel_matrix(symbolic=True)

t4pPC.centerTitle("Energies")
ethene.summary()

t4pPC.centerTitle("Energy diagram without frontier orbitals")
ethene.plot_energy_diagram()

t4pPC.centerTitle("Energy diagram with frontier orbitals")
ethene.plot_energy_diagram("FMO")

t4pPC.centerTitle("pi population")
ethene.plot_pi_population()


### [H.2] Système allyle

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.2]</b></span></td></tr></table>

On veut déterminer, grâce à la bibliothèque `hmo` des `pyphyschemtools`,  les OM &pi; du radical allyle ainsi que leur énergie par la méthode de Hückel. C'est une molécule **plane**.

1.  Recherchez sur [PubChem](https://pubchem.ncbi.nlm.nih.gov/) le code SMILES de l'allyle. Instanciez une session HMO avec l'allyle, et utilisez la fonction `show` avec l'option `show_hybridization=True`. Combien cette molécule possède-t-elle d'électrons &pi;?
2.  Écrivez l'hamiltonien Hückel de l'allyle. Vérifiez avec la bibliothèque `hmo`.
3.  Tracez le diagramme d'OM &pi;.
4.  Affichez toutes les OM &pi;. Analysez leur nature liante, antiliante ou bien nonliante.
5.  L'OM &Pi;<sub>2</sub> a pour expression &Pi;<sub>2</sub> = *C* &times; (2p<sub>z<sub>0</sub></sub> + 2p<sub>z<sub>1</sub></sub>), avec *C* &approx; 0.707. Vrai ou faux ? Vérifiez avec la fonction appropriée de `HMO`.
6.  À l'aide des résultats précédents, calculez l'énergie totale &pi; ainsi que l'énergie d'atomisation.
7.  Vérifiez ces deux valeurs, par exemple avec la fonction `summary()` de `HMO`.

</div>

In [ ]:
# Entrez le code python



### [H.3] Molécule de butadiène

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.3]</b></span></td></tr></table>

On veut déterminer, grâce à la bibliothèque `hmo` des `pyphyschemtools`, les OM &pi; du buta-1,3-diène ainsi que leurs énergies par la méthode de Hückel. C'est une molécule **plane**.

1. Recherchez sur [PubChem](https://pubchem.ncbi.nlm.nih.gov/) le code SMILES du butadiène. Instanciez une session HMO avec ce SMILES, et utilisez la fonction `show` avec l'option `show_hybridization=True`. Combien cette molécule possède-t-elle d'électrons &pi; ?
2. Écrivez l'hamiltonien de Hückel du butadiène. Vérifiez avec la bibliothèque `hmo`.
3. Tracez le diagramme d'OM &pi;.
4. Affichez toutes les OM &pi;. Analysez leur nature liante, antiliante ou non liante.
5. Calculez les indices de liaison &pi; des liaisons C0–C1, C1–C2 et C2–C3, comme dans l'exemple de l'éthylène. Les liaisons du butadiène sont-elles toutes simples ou doubles ? Par quel schéma remplaceriez-vous la notation H<sub>2</sub>C=CH–CH=CH<sub>2</sub> pour rendre compte de ces indices ? Comment appelle-t-on cette représentation ?
6. Comparez les indices de liaison aux longueurs expérimentales des liaisons C–C du butadiène, ainsi qu'aux longueurs usuelles des liaisons C–C simples et doubles.
7. Calculez l'énergie d'atomisation à l'aide des résultats précédents, puis vérifiez la valeur avec `HMO`.

<u>*Données*</u> :
- longueurs expérimentales dans le butadiène : *d*(C0–C1) = *d*(C2–C3) = 1,34 Å ; *d*(C1–C2) = 1,47 Å ;
- longueurs usuelles : *d*(C–C) = 1,54 Å (éthane) ; *d*(C=C) = 1,34 Å (éthylène).

</div>

In [ ]:
# Entrez le code python



### [H.4] Pyrrole et pyridine

<div class="intro">

<table width="100%" cellpadding="6"><tr><td bgcolor="#e6e6e6"><span style="color:#000000">📖 <b>Introduction</b></span></td></tr></table>

Dans un hétérocycle, l'hétéroatome n'a pas la même électronégativité que le carbone : son intégrale coulombienne s'écrit $\alpha_X = \alpha + h_X\,\beta$, et l'intégrale de résonance de ses liaisons $\beta_{CX} = k_{CX}\,\beta$. Les paramètres $h_X$ et $k_{CX}$ sont tabulés ; la classe `HMO` les choisit automatiquement selon la nature de l'hétéroatome, et indique aussi combien d'électrons π chaque atome apporte.

On compare ici le pyrrole et la pyridine au benzène, pour comprendre comment l'azote modifie la répartition des électrons π et la réactivité du cycle.

</div>
<br>
<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.4]</b></span></td></tr></table>

Créez une session `HMO` pour le benzène (`C1=CC=CC=C1`), le pyrrole (`C1=CC=CN1`) et la pyridine (`C1=CC=NC=C1`). Repérez la numérotation des atomes avec `show()`.

1. **Paramètres de l'azote.** Affichez les paramètres utilisés avec la méthode `used_parameters()`. Quelles sont les valeurs de $h_N$ et de $k_{CN}$ pour chaque azote ? Combien d'électrons π apporte chacun ? Pourquoi ces valeurs diffèrent-elles entre le pyrrole et la pyridine ?
2. **Hamiltonien.** Affichez l'hamiltonien de Hückel du pyrrole sous forme symbolique. Où apparaissent les paramètres de l'azote ?
3. **Charges π.** Affichez la population π des trois molécules avec `plot_pi_population()`. Quel est le signe de la charge π portée par l'azote dans le pyrrole ? Dans la pyridine ? Interprétez : dans quel cas l'azote cède-t-il des électrons au cycle, dans quel cas en attire-t-il ?
4. **Orbitales frontalières.** Comparez les énergies de la HO et de la BV des trois molécules (méthode `properties()`, ou diagramme d'OM).
    - Lequel des trois cycles est le meilleur **nucléophile** (le plus réactif vis-à-vis d'un électrophile) ?
    - Lequel est le meilleur **électrophile** (le plus réactif vis-à-vis d'un nucléophile) ?
5. **Régiosélectivité.** Expérimentalement, le pyrrole subit les substitutions électrophiles en position 2, c'est-à-dire sur un carbone voisin de l'azote.
    - Sur quels carbones les charges π sont-elles les plus négatives ? Cette prévision est-elle correcte ?
    - Tracez le diagramme d'OM avec l'option `"FMO"`. Sur quels carbones les coefficients de la HO sont-ils les plus grands ? Cette prévision est-elle correcte ?
    - Pour la pyridine, sur quels carbones un nucléophile attaquera-t-il préférentiellement ?

</div>

In [ ]:
# Entrez le code python



### [H.5] Structure électronique de polyènes monocycliques conjugués plans C<sub><i>n</i></sub>H<sub><i>n</i></sub>. Règle de Hückel

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.5]</b></span></td></tr></table>

Ces polyènes sont supposés plans. La règle de Hückel s'énonce ainsi : pour *n* pair, deux cas se présentent (*m* entier > 0) :
- si *n* = 4*m*, on a des **biradicaux** libres instables, à couches ouvertes : les HO sont non liantes ;
- si *n* = 4*m* + 2, on a des molécules **stables**, à couches fermées : toutes les OM occupées sont liantes.

1. Écrivez le SMILES des polyènes C<sub><i>n</i></sub>H<sub><i>n</i></sub> pour *n* = 4, 6, 8, 10, 12 et 14, en alternant simples et doubles liaisons. Construisez-les avec une boucle, à partir d'une chaîne de caractères. Si besoin, demandez à une IA générative de vous aider à écrire cette boucle, et vérifiez le résultat en dessinant les molécules avec la fonction `show()`.
2. Pour chaque polyène, créez une session `HMO` et tracez son diagramme d'OM &pi;, avec les OM frontalières ("FMO").
3. Les conclusions que l'on peut tirer de l'occupation des OM par les électrons &pi; concordent-elles avec la règle de Hückel ?
4. Calculez l'énergie d'atomisation par électron &pi; de chaque polyène, en unités de |&beta;|, et tracez-la en fonction du nombre d'électrons &pi;. Ajoutez sur le graphe la valeur obtenue pour une double liaison isolée (éthylène). Discutez à la lumière de la règle de Hückel.
5. Le cyclooctatétraène (COT) C<sub>8</sub>H<sub>8</sub> n'est pas plan. Visualisez sa géométrie en exécutant le premier code ci-dessous. Est-ce un résultat surprenant au vu des résultats précédents ?
6. Le COT peut se complexer avec le cérium, un lanthanide (*Z* = 58). On dispose des informations suivantes :
    - dans le complexe, le COT adopte une géométrie plane ;
    - la molécule sandwich (COT)<sub>2</sub>Ce adopte une géométrie où le cérium est stabilisé par deux cycles COT parallèles ;
    - les lanthanides ont tendance à céder leurs électrons lorsqu'ils se complexent.

    Affichez la configuration électronique fondamentale du cérium avec la bibliothèque `mendeleev`. Combien d'électrons de valence possède-t-il ? Déduisez de toutes ces informations un nombre d'oxydation plausible du cérium.

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

<u>**algorithmique basique**</u>
Une chaîne de caractères se répète avec `*` et se concatène avec `+` : `"C=C" * 2` donne `"C=CC=C"`. Le SMILES d'un cycle s'ouvre et se ferme avec le même chiffre, par exemple `"C1=CC=C1"` pour le cyclobutadiène.


<u>**Configuration électronique fondamentale (*CEF*) d'un élément.**</u> La bibliothèque `mendeleev` donne directement la configuration électronique fondamentale d'un élément, à partir de son symbole :

```python
from mendeleev import element
print(element("H").econf)
```

**Attention à la notation** : une sous-couche écrite sans exposant contient **un** électron. Par exemple, `1s` signifie 1s<sup>1</sup>.


</div>

In [ ]:
# Démo
## Géométrie 3D du COT

from pyphyschemtools import molView
path2xyz = get_data("Molecules-data/cottub.mol")
mv = molView(path2xyz)

In [ ]:
# Entrez le code python



### [H.6] Symétrie des OM &pi; de l'éthylène

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.6]</b></span></td></tr></table>

L'exécution de la cellule Python suivante répond aux questions ci-dessous. Analysez le code ainsi que les résultats et faites le lien avec les questions. 

1. Quel est le code SMILES de l'éthylène ?
2. Déterminez, à l'aide de `HMO`, la structure électronique &pi; de l'éthylène. Affichez le diagramme d'OM &pi; et les orbitales frontalières ("FMO").
3. Créez par ailleurs un objet `ethene_sym` du module `symmetry`, à partir du SMILES de l'éthylène. Quel est son groupe ponctuel ?
4. Dans quel plan se trouve la molécule ? En déduire quelle orbitale p des carbones (p<sub>x</sub>, p<sub>y</sub> ou p<sub>z</sub>) participe au système &pi;.
5. La méthode `gamma_orbitals()` indique la symétrie des combinaisons que l'on peut former avec ces orbitales. Combien de représentations irréductibles obtient-on ? Est-ce cohérent avec le nombre d'OM &pi; trouvées par `HMO` ?
6. Attribuez une représentation irréductible à chacune des deux OM &pi; calculées par `HMO`.
7. Laquelle est l'OM liante &pi;, laquelle est l'OM antiliante &pi;<sup>*</sup> ? Comment le comportement de chacune par rapport au centre d'inversion (indice g ou u) se lit-il sur le dessin des OM ?

</div>
<br>
<div class="hint">

<table width="100%" cellpadding="6"><tr><td bgcolor="#ededb6"><span style="color:#8B4513">💡 <b>Indice(s)</b></span></td></tr></table>

- Diagramme et orbitales frontalières : `h.plot_energy_diagram(mode="FMO")`.
- Plan de la molécule : `ethene_sym.orientation()`.
- Base des orbitales p<sub>x</sub> des carbones (à adapter selon la question 3) : `ethene_sym.gamma_orbitals({"C": "px"})`.
- Symétrie des OM de `HMO` : `ethene_sym.orbital_symmetries(h.c_MOs, h.E_MOs, atoms=[0, 1], shell="px")`. Les deux carbones sont les atomes 0 et 1, dans les deux objets.

</div>

In [ ]:
# Démo

from pyphyschemtools.huckel import HMO
from pyphyschemtools.symmetry import MolSym
from pyphyschemtools.Chem3D import molView

smiles = "C=C"
#################################################################################
t4pPC.centerTitle("1. Structure électronique π (HMO)")
h = HMO(smiles)
h.plot_energy_diagram(mode="FMO")

#################################################################################
t4pPC.centerTitle("2. Groupe ponctuel")
ethene = molView(smiles, source="smiles", display_now=True, show_n=True)
ethene_sym = MolSym.from_molView(ethene, lang="Fr")
print(ethene_sym.group)

#################################################################################
t4pPC.centerTitle("3. Orientation et orbitales π")
ethene_sym.orientation()
display(Markdown(r"""
La molécule est dans le plan *yz*, avec la liaison C=C selon *z*. Le système π est donc formé des orbitales **p<sub>x</sub>**, perpendiculaires au plan de la molécule.
"""))

#################################################################################
t4pPC.centerTitle("4. Symétrie des combinaisons de p<sub>x</sub>")
ethene_sym.gamma_orbitals({"C": "px"})
display(Markdown(r"""
Γ = B<sub>2g</sub> + B<sub>3u</sub> : deux représentations irréductibles, pour deux orbitales p<sub>x</sub>. C'est cohérent avec les deux OM π de HMO.
"""))

#################################################################################
t4pPC.centerTitle("5. Symétrie des OM π de HMO")
ethene_sym.orbital_symmetries(h.c_MOs, h.E_MOs, atoms=[0, 1], shell="px")

#################################################################################
t4pPC.centerTitle("6. Interprétation")
display(Markdown(r"""
- *x* = 1 (E = α + β) : OM liante **π**, (p<sub>x</sub>(C1) + p<sub>x</sub>(C2))/√2, de symétrie **B<sub>3u</sub>**. Par l'inversion, chaque lobe + est envoyé sur un lobe − : l'OM change de signe, elle est **u**.
- *x* = −1 (E = α − β) : OM antiliante **π<sup>*</sup>**, (p<sub>x</sub>(C1) − p<sub>x</sub>(C2))/√2, de symétrie **B<sub>2g</sub>**. Les deux changements de signe (orbitale p et coefficient) se compensent : l'OM est inchangée, elle est **g**.
"""))

<div class="ex">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">📝 <b>Exercice [H.7]</b></span></td></tr></table>

Retrouvez "sur le papier" la symétrie des OM &pi; de l'éthylène.
</div>

### [H.7] Bac à sable

<div class="app">

<table width="100%" cellpadding="6"><tr><td bgcolor="#d8edf4"><span style="color:#0D47A1">🚀 <b>Application</b></span></td></tr></table>

Créez ici vos propres exemples d'application
</div>

In [ ]:
# Entrez le code python



## [I] Fin